In [2]:
#cutoff:1-47/48-77
from pathlib import Path
import csv

In [3]:
#import txt檔案
input_path = (
    Path.home()
    / "Desktop"
    / "summerintern"
    / "10_genotyping_data.txt"
)

output_folder = (
    Path.home()
    / "Desktop"
    / "summerintern"
    / "split_data"
)

output_folder.mkdir(parents=True, exist_ok=True)

In [4]:
#fornt_path:1/47
#back_path:48-77

front_path = output_folder / "genotype_front_1_47.txt"
back_path = output_folder / "genotype_back_48_77.txt"
metadata_path = output_folder / "metadata.txt"
report_path = output_folder / "split_report.txt"

In [5]:
import csv

# 設定原始 TXT 每一列理論上應有的欄位數
expected_columns = 77

# 設定前段要保留到第 47 欄
# Python 切片不包含結束位置，因此 row[:47] 會取得第 1–47 欄
front_end = 47

# 顯示設定結果，確認沒有輸入錯誤
print("每列預期欄位數：", expected_columns)
print("前段欄位數：", front_end)
print("後段欄位數：", expected_columns - front_end)

每列預期欄位數： 77
前段欄位數： 47
後段欄位數： 30


In [6]:
# metadata 列數
metadata_count = 0

#記錄表頭與 SNP 資料列數
data_count = 0

# 記錄欄位數不是 77 的異常列
bad_rows = []


In [7]:
# 同時開啟原始 TXT、前段輸出檔、後段輸出檔與 metadata 輸出檔
with input_path.open(
    "r",                    
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 遇到無法解碼的字元時，用替代符號取代
    newline=""              # 保留原始換行處理方式
) as infile, \
front_path.open(
    "w",                    # 以寫入模式建立前段檔案
    encoding="utf-8",
    newline=""
) as front_file, \
back_path.open(
    "w",                    # 以寫入模式建立後段檔案
    encoding="utf-8",
    newline=""
) as back_file, \
metadata_path.open(
    "w",                    # 以寫入模式建立 metadata 檔案
    encoding="utf-8",
    newline=""
) as metadata_file:

    # 建立 Tab 分隔的讀取器
    reader = csv.reader(
        infile,
        delimiter="\t",     # 指定 Tab 為欄位分隔符號
        quoting=csv.QUOTE_NONE
    )

    # 建立前段檔案的寫入器
    front_writer = csv.writer(
        front_file,
        delimiter="\t",     # 輸出仍使用 Tab 分隔
        quoting=csv.QUOTE_NONE,
        escapechar="\\",    # 特殊字元使用反斜線處理
        lineterminator="\n" # 每列結尾使用換行符號
    )

    # 建立後段檔案的寫入器
    back_writer = csv.writer(
        back_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 逐列讀取原始 TXT
    for physical_line_number, row in enumerate(reader, start=1):

        # 如果該列以 ## 開頭，代表是 metadata
        if row and row[0].startswith("##"):

            # 將 metadata 原樣寫入 metadata.txt
            metadata_file.write("\t".join(row) + "\n")

            # metadata 列數加 1
            metadata_count += 1

            # 跳過後面的切割步驟，直接處理下一列
            continue

        # 記錄表頭與 SNP 資料列數
        data_count += 1

        # 檢查該列欄位數是否為 77
        if len(row) != expected_columns:

            # 記錄異常列的實體列號、欄位數與第一欄內容
            bad_rows.append(
                (
                    physical_line_number,
                    len(row),
                    row[0] if row else ""
                )
            )

        # 將第 1–47 欄寫入前段檔案
        front_writer.writerow(row[:front_end])

        # 將第 48–77 欄寫入後段檔案
        back_writer.writerow(row[front_end:])

# 顯示切割結果
print("切割完成")
print("metadata 列數：", metadata_count)
print("表頭與資料列數：", data_count)
print("欄位數不是 77 的列數：", len(bad_rows))
print("前段檔案：", front_path)
print("後段檔案：", back_path)
print("metadata 檔案：", metadata_path)

切割完成
metadata 列數： 5
表頭與資料列數： 680866
欄位數不是 77 的列數： 0
前段檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_front_1_47.txt
後段檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_48_77.txt
metadata 檔案： /Users/sheena/Desktop/summerintern/split_data/metadata.txt


In [8]:
# 讀取前段檔案的第一列 欄位名稱
with front_path.open(
    "r",                    
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace"        # 無法解碼的字元以替代符號顯示
) as front_file:

    # 讀取第一列
    front_header_line = front_file.readline()

    # 移除換行符號，再用 Tab 切成欄位清單
    front_header = front_header_line.rstrip("\r\n").split("\t")


# 讀取後段檔案的第一列，也就是欄位名稱
with back_path.open(
    "r",                    # 以讀取模式開啟後段檔案
    encoding="utf-8",
    errors="replace"
) as back_file:

    # 讀取第一列
    back_header_line = back_file.readline()

    # 移除換行符號，再用 Tab 切成欄位清單
    back_header = back_header_line.rstrip("\r\n").split("\t")


# 顯示前段檢查結果
print("前段欄位數：", len(front_header))
print("前段第一欄：", front_header[0])
print("前段最後一欄：", front_header[-1])

# 空一行，讓輸出比較容易閱讀
print()

# 顯示後段檢查結果
print("後段欄位數：", len(back_header))
print("後段第一欄：", back_header[0])
print("後段最後一欄：", back_header[-1])

前段欄位數： 47
前段第一欄： probeset_id
前段最後一欄： CR

後段欄位數： 30
後段第一欄： FLD
後段最後一欄： Call Modified


In [9]:
#找出後段資料中常見的排列模式
# 建立空字典，用來統計後段資料中各種「欄位內容模式」出現的次數
pattern_counts = {}

# genotype_back_48_77.txt
with back_path.open(
    "r",                    # 以讀取模式開啟檔案
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 遇到無法解碼的字元時，用替代符號取代
    newline=""              # 保留原始換行處理
) as back_file:

    # 建立以 Tab 為分隔符號的讀取器
    reader = csv.reader(
        back_file,
        delimiter="\t",     # 指定 Tab 為欄位分隔符號
        quoting=csv.QUOTE_NONE
    )

    # 讀取第一列欄位名稱
    back_header = next(reader)

    # 逐列讀取後段 SNP 資料
    for row_number, row in enumerate(reader, start=1):

        # 如果該列欄位不足 30 欄，先補空字串
        # 這樣後面取固定位置時不會發生 IndexError
        if len(row) < 30:
            row = row + [""] * (30 - len(row))

        # 取出幾個具有辨識力的欄位內容
        # 後段第 9 欄原本對應 n_BB
        n_BB_value = row[8]

        # 後段第 10 欄原本對應 n_NC
        n_NC_value = row[9]

        # 後段第 11 欄原本對應 hemizygous
        hemizygous_value = row[10]

        # 後段第 12 欄原本對應 specialSNP_chr
        special_chr_value = row[11]

        # 後段第 13 欄原本對應 gender_metrics
        gender_value = row[12]

        # 後段第 14 欄原本對應 ConversionType
        conversion_value = row[13]

        # 將這六個欄位組合成一個 pattern
        pattern = (
            n_BB_value,
            n_NC_value,
            hemizygous_value,
            special_chr_value,
            gender_value,
            conversion_value
        )

        # 如果這個 pattern 第一次出現，就先設定次數為 0
        if pattern not in pattern_counts:
            pattern_counts[pattern] = 0

        # 該 pattern 出現次數加 1
        pattern_counts[pattern] += 1


# 將所有 pattern 依出現次數由大到小排序
sorted_patterns = sorted(
    pattern_counts.items(),
    key=lambda item: item[1],
    reverse=True
)


# 顯示前 20 種最常見的 pattern
print("最常見的 20 種後段欄位模式：")
print()

for rank, (pattern, count) in enumerate(
    sorted_patterns[:20],
    start=1
):
    print(f"第 {rank} 名，出現次數：{count}")

    print("n_BB位置：", repr(pattern[0]))
    print("n_NC位置：", repr(pattern[1]))
    print("hemizygous位置：", repr(pattern[2]))
    print("specialSNP_chr位置：", repr(pattern[3]))
    print("gender_metrics位置：", repr(pattern[4]))
    print("ConversionType位置：", repr(pattern[5]))

    print("-" * 50)

最常見的 20 種後段欄位模式：

第 1 名，出現次數：186383
n_BB位置： '0'
n_NC位置： '0'
hemizygous位置： 'autosomal'
specialSNP_chr位置： 'all'
gender_metrics位置： 'NoMinorHom'
ConversionType位置： '1'
--------------------------------------------------
第 2 名，出現次數：131659
n_BB位置： 'autosomal'
n_NC位置： 'all'
hemizygous位置： 'MonoHighResolution'
specialSNP_chr位置： '1'
gender_metrics位置： '1'
ConversionType位置： '0'
--------------------------------------------------
第 3 名，出現次數：26229
n_BB位置： '1'
n_NC位置： '0'
hemizygous位置： 'autosomal'
specialSNP_chr位置： 'all'
gender_metrics位置： 'NoMinorHom'
ConversionType位置： '1'
--------------------------------------------------
第 4 名，出現次數：17794
n_BB位置： '1'
n_NC位置： '0'
hemizygous位置： '0'
specialSNP_chr位置： 'autosomal'
gender_metrics位置： 'all'
ConversionType位置： 'PolyHighResolution'
--------------------------------------------------
第 5 名，出現次數：12292
n_BB位置： 'Y'
n_NC位置： 'male'
hemizygous位置： 'MonoHighResolution'
specialSNP_chr位置： '1'
gender_metrics位置： '1'
ConversionType位置： '0'
---------------------------------------

In [10]:
# 設定 specialSNP_chr 合理值
valid_chr_types = {"autosomal", "X", "Y", "MT"}

# 設定 gender_metrics 合理值
valid_gender_types = {"all", "male", "female"}

# 建立字典，統計不同位移量的列數
shift_counts = {}

# 建立字典，記錄無法辨識的列
unrecognized_rows = []

# 開啟後段資料
with back_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼字元以替代符號呈現
    newline=""
) as back_file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        back_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取並跳過表頭
    back_header = next(reader)

    # specialSNP_chr 在正常後段資料中應位於第 12 欄
    # Python index 從 0 開始，所以位置是 11
    expected_special_chr_index = 11

    # 逐列檢查
    for row_number, row in enumerate(reader, start=1):

        # 若不足 30 欄，補空字串
        if len(row) < 30:
            row = row + [""] * (30 - len(row))

        # 找出此列中 autosomal、X、Y 或 MT 出現的位置
        chr_positions = [
            index
            for index, value in enumerate(row)
            if value in valid_chr_types
        ]

        # 只接受剛好找到一個 chromosome type 的列
        if len(chr_positions) == 1:

            # 取得 specialSNP_chr 實際出現位置
            actual_chr_index = chr_positions[0]

            # 計算向左位移幾欄
            # 例如應在 11，實際在 10，代表向左移 1 欄
            left_shift = expected_special_chr_index - actual_chr_index

            # 檢查 chromosome 後一欄是否為合理 gender_metrics
            gender_index = actual_chr_index + 1

            if gender_index < len(row):
                gender_value = row[gender_index]
            else:
                gender_value = ""

            # 判斷後一欄是否合理
            gender_ok = gender_value in valid_gender_types

            # 以「位移量 + gender是否合理」作為分類
            pattern_key = (left_shift, gender_ok)

            # 累計該模式列數
            shift_counts[pattern_key] = (
                shift_counts.get(pattern_key, 0) + 1
            )

        else:
            # 沒找到或找到多個 chromosome type 的列，先記錄
            unrecognized_rows.append(
                (
                    row_number,
                    len(chr_positions),
                    row[:15]
                )
            )


# 依出現次數由大到小排列
sorted_shift_counts = sorted(
    shift_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

# 顯示結果
print("依 specialSNP_chr 位置判斷的位移模式：")
print()

for (left_shift, gender_ok), count in sorted_shift_counts:

    print(
        "向左位移欄數：",
        left_shift,
        "| gender位置合理：",
        gender_ok,
        "| 列數：",
        count
    )

# 顯示無法辨識列數
print()
print("無法辨識的列數：", len(unrecognized_rows))

依 specialSNP_chr 位置判斷的位移模式：

向左位移欄數： 0 | gender位置合理： True | 列數： 297913
向左位移欄數： 1 | gender位置合理： True | 列數： 223971
向左位移欄數： 3 | gender位置合理： True | 列數： 155685
向左位移欄數： 2 | gender位置合理： True | 列數： 3296

無法辨識的列數： 0


In [11]:
# 設定 specialSNP_chr 可能出現的合法值
valid_chr_types = {"autosomal", "X", "Y", "MT"}

# 設定 specialSNP_chr 在正常後段資料中的位置
# 後段第 12 欄，Python index 從 0 開始，所以是 11
expected_special_chr_index = 11

# 設定新的輸出檔案路徑
classified_back_path = (
    output_folder
    / "genotype_back_48_77_with_shift.txt"
)

# 開啟原本的後段檔案與新的分類檔案
with back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as back_file, \
classified_back_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file:

    # 建立 Tab 分隔的讀取器
    reader = csv.reader(
        back_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 建立 Tab 分隔的寫入器
    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 讀取原本後段表頭
    header = next(reader)

    # 在表頭最前面新增 left_shift
    new_header = ["left_shift"] + header

    # 寫入新的表頭
    writer.writerow(new_header)

    # 逐列讀取 SNP 後段資料
    for row_number, row in enumerate(reader, start=1):

        # 若不足 30 欄，補空字串
        if len(row) < 30:
            row = row + [""] * (30 - len(row))

        # 找出 autosomal、X、Y、MT 所在位置
        chr_positions = [
            index
            for index, value in enumerate(row)
            if value in valid_chr_types
        ]

        # 因為前一步已確認每列都只有一個合法位置
        actual_chr_index = chr_positions[0]

        # 計算向左位移欄數
        left_shift = (
            expected_special_chr_index
            - actual_chr_index
        )

        # 將 left_shift 放在每列最前面
        new_row = [left_shift] + row

        # 寫入新檔案
        writer.writerow(new_row)

# 顯示結果
print("分類檔案建立完成")
print("輸出位置：", classified_back_path)

分類檔案建立完成
輸出位置： /Users/sheena/Desktop/summerintern/split_data/genotype_back_48_77_with_shift.txt


In [11]:
# 開啟新的分類檔案
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace"
) as file:

    # 顯示前 5 列
    for i in range(5):
        line = file.readline()
        print(line.rstrip("\n"))

left_shift	FLD	HomFLD	HetSO	HomRO	nMinorAllele	Nclus	n_AA	n_AB	n_BB	n_NC	hemizygous	specialSNP_chr	gender_metrics	ConversionType	BestProbeset	BestandRecommended	HomHet	AA.meanX	AA.meanY	AB.meanX	AB.meanY	BB.meanX	BB.meanY	NC.meanX	NC.meanY	MMD	MinorAlleleFrequency	H.W.p-Value	H.W.chisquared.statistic	Call Modified
0	16.624	34.486	0.728	2.547	211	3	64	141	35	0	0	autosomal	all	PolyHighResolution	1	1	0	2.5472	9.6674	0.0158	10.3304	-2.7043	9.5337	NA	NA	41.545	0.44	0.00287	8.8852	FALSE
0	14.758	30.536	0.776	2.562	168	3	102	108	30	0	0	autosomal	all	PolyHighResolution	1	1	0	2.6958	9.4151	0.1546	10.1287	-2.5623	9.2856	NA	NA	50.182	0.35	0.865	0.029	FALSE
0	13.761	28.393	0.539	2.577	234	3	59	116	65	0	0	autosomal	all	PolyHighResolution	1	1	0	2.577	10.6359	-0.3268	11.3084	-3.0579	10.8953	NA	NA	50.462	0.488	0.612	0.2571	FALSE
0	14.287	29.199	0.571	1.952	236	3	60	124	56	0	0	autosomal	all	PolyHighResolution	1	1	0	1.9521	9.6865	-0.3371	10.3143	-2.5304	9.7971	NA	NA	61.263	0.492	0.602	0.2713	FALSE


In [12]:
# 建立字典：每種 left_shift 最多保存 3 列範例
examples_by_shift = {
    0: [],
    1: [],
    2: [],
    3: []
}

# 開啟已加入 left_shift 的後段檔案
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取表頭
    header = next(reader)

    # 逐列讀取資料
    for row_number, row in enumerate(reader, start=1):

        # 第 1 欄是 left_shift
        left_shift = int(row[0])

        # 每一類最多保留 3 列
        if len(examples_by_shift[left_shift]) < 3:
            examples_by_shift[left_shift].append(
                (row_number, row)
            )

        # 如果 4 種類型都已經各收集 3 列，就停止讀取
        if all(
            len(examples_by_shift[shift]) == 3
            for shift in examples_by_shift
        ):
            break


# 逐一顯示 4 種 left_shift 的範例
for shift in [0, 1, 2, 3]:

    print("=" * 70)
    print("left_shift =", shift)
    print("=" * 70)

    # 顯示這一類的 3 列範例
    for row_number, row in examples_by_shift[shift]:

        print("資料列號：", row_number)

        # 從第 2 欄開始，因為第 1 欄是新增的 left_shift
        for column_name, value in zip(
            header[1:],
            row[1:]
        ):
            print(
                f"{column_name}: {repr(value)}"
            )

        print("-" * 70)

left_shift = 0
資料列號： 1
FLD: '16.624'
HomFLD: '34.486'
HetSO: '0.728'
HomRO: '2.547'
nMinorAllele: '211'
Nclus: '3'
n_AA: '64'
n_AB: '141'
n_BB: '35'
n_NC: '0'
hemizygous: '0'
specialSNP_chr: 'autosomal'
gender_metrics: 'all'
ConversionType: 'PolyHighResolution'
BestProbeset: '1'
BestandRecommended: '1'
HomHet: '0'
AA.meanX: '2.5472'
AA.meanY: '9.6674'
AB.meanX: '0.0158'
AB.meanY: '10.3304'
BB.meanX: '-2.7043'
BB.meanY: '9.5337'
NC.meanX: 'NA'
NC.meanY: 'NA'
MMD: '41.545'
MinorAlleleFrequency: '0.44'
H.W.p-Value: '0.00287'
H.W.chisquared.statistic: '8.8852'
Call Modified: 'FALSE'
----------------------------------------------------------------------
資料列號： 2
FLD: '14.758'
HomFLD: '30.536'
HetSO: '0.776'
HomRO: '2.562'
nMinorAllele: '168'
Nclus: '3'
n_AA: '102'
n_AB: '108'
n_BB: '30'
n_NC: '0'
hemizygous: '0'
specialSNP_chr: 'autosomal'
gender_metrics: 'all'
ConversionType: 'PolyHighResolution'
BestProbeset: '1'
BestandRecommended: '1'
HomHet: '0'
AA.meanX: '2.6958'
AA.meanY: '9.4151'
AB.

In [13]:
# 指定這次要查看的位移類型
target_shift = 1

# 指定最多顯示幾筆範例
max_examples = 3

# 建立計數器，記錄目前已顯示幾筆
shown_count = 0

# 開啟已加入 left_shift 的後段檔案
with classified_back_path.open(
    "r",                    # 以讀取模式開啟
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 無法解碼的字元以替代符號顯示
    newline=""
) as file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取第一列欄位名稱
    header = next(reader)

    # 逐列讀取資料
    for row_number, row in enumerate(reader, start=1):

        # 第一欄是新增的 left_shift
        left_shift = int(row[0])

        # 只處理 left_shift = 1 的資料列
        if left_shift != target_shift:
            continue

        # 顯示資料列號
        print("=" * 70)
        print("資料列號：", row_number)
        print("left_shift：", left_shift)
        print("=" * 70)

        # 顯示原始後段 30 欄
        # header[1:] 跳過新增的 left_shift 欄
        # row[1:] 跳過該列的 left_shift 值
        for column_name, value in zip(
            header[1:],
            row[1:]
        ):
            print(f"{column_name}: {repr(value)}")

        # 每顯示一筆，計數器加 1
        shown_count += 1

        # 顯示滿 3 筆後停止
        if shown_count >= max_examples:
            break

# 顯示總共輸出幾筆
print()
print("已顯示範例數：", shown_count)

資料列號： 89
left_shift： 1
FLD: '4.462'
HomFLD: '?�數??-0.0327'
HetSO: '1.69'
HomRO: '13'
nMinorAllele: '2'
Nclus: '0'
n_AA: '13'
n_AB: '226'
n_BB: '1'
n_NC: '0'
hemizygous: 'autosomal'
specialSNP_chr: 'all'
gender_metrics: 'NoMinorHom'
ConversionType: '1'
BestProbeset: '1'
BestandRecommended: '1'
HomHet: '2.0892'
AA.meanX: '11.255'
AA.meanY: '-0.0631'
AB.meanX: '11.1512'
AB.meanY: '-1.6896'
BB.meanX: '11.1302'
BB.meanY: 'NA'
NC.meanX: 'NA'
NC.meanY: '?�數??0.0272'
MMD: '1'
MinorAlleleFrequency: 'NA'
H.W.p-Value: 'FALSE'
H.W.chisquared.statistic: ''
Call Modified: ''
資料列號： 91
left_shift： 1
FLD: '9.579'
HomFLD: '?�數??0.403'
HetSO: '1.612'
HomRO: '3'
nMinorAllele: '2'
Nclus: '0'
n_AA: '3'
n_AB: '237'
n_BB: '0'
n_NC: '0'
hemizygous: 'autosomal'
specialSNP_chr: 'all'
gender_metrics: 'NoMinorHom'
ConversionType: '1'
BestProbeset: '1'
BestandRecommended: '1'
HomHet: '2.4153'
AA.meanX: '9.8751'
AA.meanY: '0.4429'
AB.meanX: '10.1739'
AB.meanY: '-1.6123'
BB.meanX: '9.6633'
BB.meanY: 'NA'
NC.meanX: 'N

In [14]:
#單獨看HomFLD
# 建立空字典，用來統計 left_shift = 1 時
# HomFLD 欄位中每種原始值出現的次數
homfld_counts_shift1 = {}

# 開啟已加入 left_shift 分類的後段檔案
with classified_back_path.open(
    "r",                    # 以讀取模式開啟
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 無法解碼的字元用替代符號顯示
    newline=""
) as file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取表頭
    header = next(reader)

    # 找出 HomFLD 在目前檔案中的欄位位置
    # 因為前面多了一欄 left_shift，所以直接用欄名找最安全
    homfld_index = header.index("HomFLD")

    # 找出 left_shift 欄位位置
    shift_index = header.index("left_shift")

    # 逐列讀取資料
    for row in reader:

        # 只保留 left_shift = 1 的資料列
        if row[shift_index] != "1":
            continue

        # 取出目前 HomFLD 欄位的原始內容
        homfld_value = row[homfld_index]

        # 統計每個值出現次數
        homfld_counts_shift1[homfld_value] = (
            homfld_counts_shift1.get(homfld_value, 0) + 1
        )

# 依出現次數由多到少排序
sorted_homfld = sorted(
    homfld_counts_shift1.items(),
    key=lambda item: item[1],
    reverse=True
)

# 顯示前 30 種最常見的 HomFLD 原始值
print("left_shift = 1 中最常見的 HomFLD 值：")
print()

for value, count in sorted_homfld[:30]:
    print("出現次數：", count, "| HomFLD：", repr(value))

left_shift = 1 中最常見的 HomFLD 值：

出現次數： 613 | HomFLD： '?�數??0.329'
出現次數： 601 | HomFLD： '?�數??0.354'
出現次數： 584 | HomFLD： '?�數??0.331'
出現次數： 583 | HomFLD： '?�數??0.301'
出現次數： 583 | HomFLD： '?�數??0.287'
出現次數： 581 | HomFLD： '?�數??0.345'
出現次數： 581 | HomFLD： '?�數??0.338'
出現次數： 579 | HomFLD： '?�數??0.306'
出現次數： 579 | HomFLD： '?�數??0.361'
出現次數： 576 | HomFLD： '?�數??0.255'
出現次數： 575 | HomFLD： '?�數??0.377'
出現次數： 572 | HomFLD： '?�數??0.266'
出現次數： 567 | HomFLD： '?�數??0.262'
出現次數： 566 | HomFLD： '?�數??0.375'
出現次數： 566 | HomFLD： '?�數??0.347'
出現次數： 566 | HomFLD： '?�數??0.322'
出現次數： 564 | HomFLD： '?�數??0.359'
出現次數： 563 | HomFLD： '?�數??0.356'
出現次數： 562 | HomFLD： '?�數??0.303'
出現次數： 562 | HomFLD： '?�數??0.278'
出現次數： 560 | HomFLD： '?�數??0.269'
出現次數： 559 | HomFLD： '?�數??0.312'
出現次數： 559 | HomFLD： '?�數??0.292'
出現次數： 558 | HomFLD： '?�數??0.264'
出現次數： 557 | HomFLD： '?�數??0.326'
出現次數： 556 | HomFLD： '?�數??0.324'
出現次數： 554 | HomFLD： '?�數??0.35'
出現次數： 552 | HomFLD： '?�數??0.296'
出現次數： 551 | HomFLD： '?�數??0.28'
出現次數： 547 | H

In [15]:
#步驟（針對位移一個的）：
#1.先把FLD欄位之後 從HomFLD開始都先位移一個 這樣先空出HomFLD
#2.Backward 5個切割FLD 
#3.放入FLD

#  載入正規表示式模組
# 用來從含亂碼的 FLD 字串中擷取數字
import re
# 設定 left_shift = 1 修正版的輸出檔案
shift1_fixed_path = (
    output_folder
    / "genotype_back_shift1_fixed.txt"
)


# 用來記錄修復結果
total_shift1 = 0              # left_shift = 1 的總列數
success_count = 0             # 成功辨認 FLD 與 HomFLD 的列數
failed_count = 0              # 無法辨認兩個數值的列數
last_column_not_empty = 0     # 原始最後一欄不為空的列數

# 儲存前幾筆修復失敗的範例
failed_examples = []

# 儲存原始最後一欄不為空的範例
last_column_examples = []


# 這個 pattern 可以辨認：
# 4.462
# -0.0327
# 0.329
# .521
# -3
number_pattern = re.compile(
    r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
)


# 開啟已加入 left_shift 的後段檔案
with classified_back_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼的字元以替代符號保留
    newline=""
) as input_file, \
shift1_fixed_path.open(
    "w",                    # 建立新的修正版檔案
    encoding="utf-8",
    newline=""
) as output_file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 建立 Tab 分隔寫入器
    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 讀取欄位名稱
    header = next(reader)

    # 找出 left_shift 欄的位置
    shift_index = header.index("left_shift")

    # 後段原始 30 欄的欄位名稱
    back_columns = header[1:]

    # 輸出欄位：
    # left_shift + 修正後30欄 + repair_status
    output_header = (
        ["left_shift"]
        + back_columns
        + ["repair_status"]
    )

    # 寫入表頭
    writer.writerow(output_header)


    # 逐列讀取資料
    for row_number, row in enumerate(reader, start=1):

        # 只處理 left_shift = 1
        if row[shift_index] != "1":
            continue

        total_shift1 += 1

        # 取出原始後段30欄
        # 排除最前面的 left_shift
        original = row[1:]

        # 若不足30欄，補空字串
        if len(original) < 30:
            original = original + [""] * (30 - len(original))

        # 若超過30欄，只保留前30欄
        original = original[:30]

        # 保存原始 FLD 字串
        original_fld = original[0]

        # 從 FLD 字串中找出所有數值
        numbers = number_pattern.findall(original_fld)


        # 右移會捨棄原始最後一欄
        # 因此先確認原始 Call Modified 是否為空
        if original[-1].strip() != "":
            last_column_not_empty += 1

            if len(last_column_examples) < 10:
                last_column_examples.append(
                    (
                        row_number,
                        original[-1]
                    )
                )


        # 必須至少辨認到兩個數值：
        # 第一個視為真正 FLD
        # 最後一個視為真正 HomFLD
        if len(numbers) >= 2:

            corrected_fld = numbers[0]
            corrected_homfld = numbers[-1]

            # 先複製原始資料
            corrected = original.copy()

            # 將原本 HomFLD 到倒數第二欄
            # 全部向右移一格
            #
            # corrected[2:] 代表新的 HetSO 到 Call Modified
            # original[1:-1] 代表原始 HomFLD 到倒數第二欄
            corrected[2:] = original[1:-1]

            # 將清理後的 FLD 放回第一欄
            corrected[0] = corrected_fld

            # 將從 FLD 尾端擷取的數值放入 HomFLD
            corrected[1] = corrected_homfld

            repair_status = "success"

            success_count += 1

        else:
            # 如果 FLD 中找不到至少兩個數值
            # 暫時保留原始資料，不進行猜測性修正
            corrected = original.copy()

            repair_status = "failed_less_than_2_numbers"

            failed_count += 1

            # 最多記錄前10筆失敗範例
            if len(failed_examples) < 10:
                failed_examples.append(
                    (
                        row_number,
                        original_fld,
                        numbers
                    )
                )


        # 寫入：
        # left_shift + 修正後30欄 + 修復狀態
        writer.writerow(
            ["1"]
            + corrected
            + [repair_status]
        )


# 顯示處理結果
print("left_shift = 1 處理完成")
print("輸出檔案：", shift1_fixed_path)
print()
print("left_shift = 1 總列數：", total_shift1)
print("成功修復列數：", success_count)
print("無法辨認列數：", failed_count)
print("原始最後一欄不為空：", last_column_not_empty)

print()
print("前幾筆無法辨認的範例：")

for example in failed_examples:
    print(example)

print()
print("原始最後一欄不為空的範例：")

for example in last_column_examples:
    print(example)

left_shift = 1 處理完成
輸出檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_fixed.txt

left_shift = 1 總列數： 223971
成功修復列數： 0
無法辨認列數： 223971
原始最後一欄不為空： 0

前幾筆無法辨認的範例：
(89, '4.462', ['4.462'])
(91, '9.579', ['9.579'])
(93, '7.143', ['7.143'])
(98, '6.14', ['6.14'])
(101, '8.359', ['8.359'])
(103, '8.798', ['8.798'])
(105, '11.547', ['11.547'])
(107, '9.647', ['9.647'])
(108, '11.59', ['11.59'])
(109, '9.041', ['9.041'])

原始最後一欄不為空的範例：


In [16]:
# 載入正規表示式工具
import re

# 定義「完整純數字」格式
# 可接受 4.462、-0.0327、12、.35
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)$"
)

# 建立分類計數器
fld_numeric_homfld_numeric = 0
fld_numeric_homfld_garbled = 0
fld_garbled_homfld_numeric = 0
fld_garbled_homfld_garbled = 0

# 保存少量 FLD 含亂碼範例
fld_garbled_examples = []

# 開啟已分類的後段資料
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取表頭
    header = next(reader)

    # 用欄名取得位置
    shift_index = header.index("left_shift")
    fld_index = header.index("FLD")
    homfld_index = header.index("HomFLD")

    # 逐列檢查
    for row_number, row in enumerate(reader, start=1):

        # 只檢查 left_shift = 1
        if row[shift_index] != "1":
            continue

        fld_value = row[fld_index].strip()
        homfld_value = row[homfld_index].strip()

        # 判斷是否為完整純數字
        fld_is_numeric = bool(
            full_number_pattern.fullmatch(fld_value)
        )

        homfld_is_numeric = bool(
            full_number_pattern.fullmatch(homfld_value)
        )

        # 分成四種組合
        if fld_is_numeric and homfld_is_numeric:
            fld_numeric_homfld_numeric += 1

        elif fld_is_numeric and not homfld_is_numeric:
            fld_numeric_homfld_garbled += 1

        elif not fld_is_numeric and homfld_is_numeric:
            fld_garbled_homfld_numeric += 1

        else:
            fld_garbled_homfld_garbled += 1

        # 保存前 20 筆 FLD 不是純數字的範例
        if not fld_is_numeric and len(fld_garbled_examples) < 20:
            fld_garbled_examples.append(
                (
                    row_number,
                    fld_value,
                    homfld_value
                )
            )

# 顯示統計
print("FLD純數字、HomFLD純數字：",
      fld_numeric_homfld_numeric)

print("FLD純數字、HomFLD含亂碼：",
      fld_numeric_homfld_garbled)

print("FLD含亂碼、HomFLD純數字：",
      fld_garbled_homfld_numeric)

print("FLD含亂碼、HomFLD含亂碼：",
      fld_garbled_homfld_garbled)

print()
print("FLD含亂碼的前20筆範例：")

for example in fld_garbled_examples:
    print(example)



FLD純數字、HomFLD純數字： 0
FLD純數字、HomFLD含亂碼： 223971
FLD含亂碼、HomFLD純數字： 0
FLD含亂碼、HomFLD含亂碼： 0

FLD含亂碼的前20筆範例：


發現位移一個是從HomFLD之後！！！

In [17]:
# 載入正規表示式模組
# 用來從 HomFLD 亂碼字串尾端擷取完整數值
import re


# 設定修正後的輸出檔案路徑
shift1_fixed_path = (
    output_folder
    / "genotype_back_shift1_fixed.txt"
)


# 設定修復檢查紀錄檔
shift1_audit_path = (
    output_folder
    / "genotype_back_shift1_audit.txt"
)


# 從字串尾端擷取完整數值
# 可辨認：
# 0.329
# -0.0327
# 12
# .35
# 1.2e-3
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立處理結果計數器
total_shift1 = 0                 # left_shift = 1 的總列數
success_count = 0                # 成功擷取 HomFLD 的列數
failed_count = 0                 # 無法擷取 HomFLD 的列數
last_column_not_empty = 0        # 原始最後一欄不是空白的列數
batch_total_240_count = 0        # 修正後 n_AA+n_AB+n_BB+n_NC = 240 的列數
batch_total_invalid_count = 0    # 修正後 genotype count 不等於 240 的列數


# 保存少量失敗範例
failed_examples = []


# 同時開啟：
# 1. 已加入 left_shift 的後段檔案
# 2. 修正後輸出檔案
# 3. 修復紀錄檔
with classified_back_path.open(
    "r",                         # 讀取模式
    encoding="utf-8",            # 使用 UTF-8
    errors="replace",            # 無法解碼字元以 � 保留
    newline=""
) as input_file, \
shift1_fixed_path.open(
    "w",                         # 寫入新的修正版
    encoding="utf-8",
    newline=""
) as output_file, \
shift1_audit_path.open(
    "w",                         # 寫入修復前後對照紀錄
    encoding="utf-8",
    newline=""
) as audit_file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 建立修正版寫入器
    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 建立修復紀錄寫入器
    audit_writer = csv.writer(
        audit_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )


    # 讀取原始表頭
    header = next(reader)

    # 取得後段原始 30 個欄位名稱
    # 跳過最前面的 left_shift
    back_columns = header[1:]


    # 修正版新增：
    # source_row_number：原始 SNP 資料列號
    # repair_status：修復是否成功
    writer.writerow(
        ["source_row_number", "left_shift"]
        + back_columns
        + ["repair_status"]
    )


    # 修復紀錄檔只保存關鍵對照資訊
    audit_writer.writerow(
        [
            "source_row_number",
            "original_FLD",
            "original_HomFLD",
            "fixed_HomFLD",
            "repair_status"
        ]
    )


    # 逐列讀取後段資料
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 第一欄是 left_shift
        # 只處理 left_shift = 1
        if row[0] != "1":
            continue

        total_shift1 += 1


        # 取得原始後段 30 欄
        # row[0] 是 left_shift，因此從 row[1] 開始
        original = row[1:]


        # 如果不足 30 欄，用空字串補足
        if len(original) < 30:
            original = original + [""] * (30 - len(original))


        # 如果意外超過 30 欄，只保留前 30 欄
        original = original[:30]


        # 原始欄位位置：
        # original[0] = FLD
        # original[1] = HomFLD
        # original[2] = HetSO
        original_fld = original[0]
        original_homfld = original[1]


        # 從原始 HomFLD 字串尾端尋找完整數值
        match = ending_number_pattern.search(
            original_homfld
        )


        # 修正時最後一欄會被移除
        # 因此先確認原始最後一欄是否為空
        if original[-1].strip() != "":
            last_column_not_empty += 1


        # 成功找到 HomFLD 尾端數值
        if match:

            # 取得真正的 HomFLD 數值
            fixed_homfld = match.group(1)


            # 建立新的 30 欄空白資料
            corrected = [""] * 30


            # FLD 保持原始值不變
            corrected[0] = original_fld


            # HomFLD 改成從亂碼字串尾端擷取的數值
            corrected[1] = fixed_homfld


            # HetSO 暫時留空
            # 因為這是此類資料缺少的欄位
            corrected[2] = ""


            # 將原始 HetSO 到倒數第二欄
            # 全部向右移一格
            #
            # 原始：
            # original[2] 放在 HetSO 位置
            #
            # 修正：
            # original[2] 移到新的 HomRO 位置 corrected[3]
            corrected[3:] = original[2:-1]


            # 標記修復成功
            repair_status = "success"
            success_count += 1


            # 修正後欄位位置：
            # n_AA = index 6
            # n_AB = index 7
            # n_BB = index 8
            # n_NC = index 9
            try:
                batch_total = (
                    int(corrected[6])
                    + int(corrected[7])
                    + int(corrected[8])
                    + int(corrected[9])
                )

                # 檢查四種 genotype count 是否合計 240
                if batch_total == 240:
                    batch_total_240_count += 1
                else:
                    batch_total_invalid_count += 1

            except ValueError:
                # 若任一 genotype count 無法轉成整數
                batch_total_invalid_count += 1


        # 無法從 HomFLD 尾端找到數值
        else:

            # 不進行猜測性修正
            corrected = original.copy()

            fixed_homfld = ""

            repair_status = "failed_no_ending_number"
            failed_count += 1


            # 最多保存前 10 筆失敗範例
            if len(failed_examples) < 10:
                failed_examples.append(
                    (
                        source_row_number,
                        original_fld,
                        original_homfld
                    )
                )


        # 寫入修正後資料
        writer.writerow(
            [
                source_row_number,
                "1"
            ]
            + corrected
            + [repair_status]
        )


        # 寫入修復前後對照紀錄
        audit_writer.writerow(
            [
                source_row_number,
                original_fld,
                original_homfld,
                fixed_homfld,
                repair_status
            ]
        )


# 顯示處理結果
print("left_shift = 1 修復完成")
print()

print("修正版檔案：", shift1_fixed_path)
print("修復紀錄檔：", shift1_audit_path)
print()

print("left_shift = 1 總列數：", total_shift1)
print("成功修復列數：", success_count)
print("無法擷取 HomFLD 列數：", failed_count)
print("原始最後一欄不為空：", last_column_not_empty)
print()

print(
    "修正後 genotype count 合計為 240：",
    batch_total_240_count
)

print(
    "修正後 genotype count 無法通過檢查：",
    batch_total_invalid_count
)

print()
print("前幾筆失敗範例：")

for example in failed_examples:
    print(example)

left_shift = 1 修復完成

修正版檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_fixed.txt
修復紀錄檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_audit.txt

left_shift = 1 總列數： 223971
成功修復列數： 223971
無法擷取 HomFLD 列數： 0
原始最後一欄不為空： 0

修正後 genotype count 合計為 240： 220896
修正後 genotype count 無法通過檢查： 3075

前幾筆失敗範例：


In [18]:
# 載入 Counter，用來統計各種異常原因與資料類型
from collections import Counter

# 統計未通過 count QC 的原因
invalid_reason_counts = Counter()

# 統計未通過者的 chromosome、gender與ConversionType組合
invalid_group_counts = Counter()

# 統計四個 genotype count 實際合計值
invalid_total_counts = Counter()

# 保存前20筆未通過的完整範例
invalid_examples = []


# 開啟剛才建立的 left_shift = 1 修正版檔案
with shift1_fixed_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取每一列
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查修正後的 genotype count
    for row in reader:

        # 取出四個 count 欄位的原始字串
        count_values = [
            row["n_AA"],
            row["n_AB"],
            row["n_BB"],
            row["n_NC"]
        ]

        try:
            # 嘗試將四個欄位轉成整數
            count_numbers = [
                int(value)
                for value in count_values
            ]

            # 計算四個 genotype count 的合計
            batch_total = sum(count_numbers)

            # 合計為240代表通過，不需要記錄
            if batch_total == 240:
                continue

            # 可以轉成整數，但合計不是240
            reason = "total_not_240"

            # 統計實際合計值
            invalid_total_counts[batch_total] += 1

        except (ValueError, TypeError):

            # 至少一個 count 是空白、NA或不能轉成整數
            reason = "non_integer_or_missing"
            batch_total = None

        # 統計異常原因
        invalid_reason_counts[reason] += 1

        # 依特殊染色體、性別指標與ConversionType分類
        group_key = (
            row["specialSNP_chr"],
            row["gender_metrics"],
            row["ConversionType"]
        )

        invalid_group_counts[group_key] += 1

        # 保存前20筆範例
        if len(invalid_examples) < 20:
            invalid_examples.append(
                {
                    "source_row_number": row["source_row_number"],
                    "n_AA": row["n_AA"],
                    "n_AB": row["n_AB"],
                    "n_BB": row["n_BB"],
                    "n_NC": row["n_NC"],
                    "batch_total": batch_total,
                    "specialSNP_chr": row["specialSNP_chr"],
                    "gender_metrics": row["gender_metrics"],
                    "ConversionType": row["ConversionType"]
                }
            )


# 顯示未通過原因
print("未通過原因：")

for reason, count in invalid_reason_counts.most_common():
    print(reason, "：", count)


# 顯示最常見的實際合計值
print()
print("最常見的 genotype count 合計值：")

for total, count in invalid_total_counts.most_common(20):
    print("合計 =", total, "| 列數 =", count)


# 顯示最常見的資料類型組合
print()
print("最常見的 chromosome／gender／ConversionType 組合：")

for group, count in invalid_group_counts.most_common(20):
    print(
        "specialSNP_chr =", group[0],
        "| gender_metrics =", group[1],
        "| ConversionType =", group[2],
        "| 列數 =", count
    )


# 顯示前20筆範例
print()
print("前20筆未通過範例：")

for example in invalid_examples:
    print(example)

未通過原因：
total_not_240 ： 3075

最常見的 genotype count 合計值：
合計 = 163 | 列數 = 3075

最常見的 chromosome／gender／ConversionType 組合：
specialSNP_chr = X | gender_metrics = female | ConversionType = PolyHighResolution | 列數 = 2696
specialSNP_chr = X | gender_metrics = female | ConversionType = NoMinorHom | 列數 = 379

前20筆未通過範例：
{'source_row_number': '216', 'n_AA': '0', 'n_AB': '11', 'n_BB': '152', 'n_NC': '0', 'batch_total': 163, 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'PolyHighResolution'}
{'source_row_number': '242', 'n_AA': '0', 'n_AB': '8', 'n_BB': '155', 'n_NC': '0', 'batch_total': 163, 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'PolyHighResolution'}
{'source_row_number': '255', 'n_AA': '153', 'n_AB': '10', 'n_BB': '0', 'n_NC': '0', 'batch_total': 163, 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'PolyHighResolution'}
{'source_row_number': '804', 'n_AA': '152', 'n_AB': '10', 'n_BB': '0', 'n_NC': '1', 'batch_total': 163, '

##位移2個##

In [19]:
# 指定這次要查看的位移類型
target_shift = 2

# 設定最多顯示幾筆範例
max_examples = 3

# 建立計數器，記錄目前已顯示幾筆
shown_count = 0


# 開啟已加入 left_shift 分類的後段檔案
with classified_back_path.open(
    "r",                    # 以讀取模式開啟檔案
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 無法解碼的字元以替代符號顯示
    newline=""
) as file:

    # 建立 Tab 分隔的讀取器
    reader = csv.reader(
        file,
        delimiter="\t",     # 指定 Tab 為欄位分隔符號
        quoting=csv.QUOTE_NONE
    )

    # 讀取第一列欄位名稱
    header = next(reader)


    # 逐列讀取資料
    for row_number, row in enumerate(
        reader,
        start=1
    ):

        # 第一欄是新增的 left_shift
        left_shift = int(row[0])

        # 只查看 left_shift = 2 的資料列
        if left_shift != target_shift:
            continue


        # 顯示這筆資料的基本資訊
        print("=" * 70)
        print("資料列號：", row_number)
        print("left_shift：", left_shift)
        print("=" * 70)


        # header[1:] 跳過 left_shift 欄位名稱
        # row[1:] 跳過該列的 left_shift 數值
        for column_name, value in zip(
            header[1:],
            row[1:]
        ):

            # repr() 可以清楚顯示空字串與亂碼
            print(
                f"{column_name}: {repr(value)}"
            )


        # 每顯示一筆，計數器加 1
        shown_count += 1


        # 顯示滿指定筆數後停止
        if shown_count >= max_examples:
            break


# 顯示這次實際輸出的範例數
print()
print("已顯示範例數：", shown_count)

資料列號： 88
left_shift： 2
FLD: '?�數???�數??0.548'
HomFLD: '3.205'
HetSO: '4'
HomRO: '2'
nMinorAllele: '159'
Nclus: '4'
n_AA: '0'
n_AB: '0'
n_BB: '0'
n_NC: 'X'
hemizygous: 'female'
specialSNP_chr: 'PolyHighResolution'
gender_metrics: '1'
ConversionType: '1'
BestProbeset: '1'
BestandRecommended: '3.2047'
HomHet: '10.7486'
AA.meanX: '0.3084'
AA.meanY: '11.334'
AB.meanX: 'NA'
AB.meanY: 'NA'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
資料列號： 141
left_shift： 2
FLD: '?�數???�數??0.43'
HomFLD: '2.379'
HetSO: '3'
HomRO: '2'
nMinorAllele: '0'
Nclus: '3'
n_AA: '160'
n_AB: '0'
n_BB: '0'
n_NC: 'X'
hemizygous: 'female'
specialSNP_chr: 'PolyHighResolution'
gender_metrics: '1'
ConversionType: '1'
BestProbeset: '1'
BestandRecommended: 'NA'
HomHet: 'NA'
AA.meanX: '0.0985'
AA.meanY: '10.1475'
AB.meanX: '-2.3795'
AB.meanY: '9.5921'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0092'
NC

利用逐一欄位檢查 可以知道這些位移兩個 共3296個 分別在每個欄位中的情形也許就可以正確改正

In [20]:
# 載入需要的工具
import csv
import re

# Counter 用來統計不同值或不同模式的出現次數
from collections import Counter

# median 用來計算數值欄位的中位數
from statistics import median


# 設定 left_shift = 2 的逐欄分析報告路徑
shift2_column_report_path = (
    output_folder
    / "shift2_column_summary.txt"
)

# 設定 left_shift = 2 的子模式報告路徑
shift2_pattern_report_path = (
    output_folder
    / "shift2_subpattern_summary.txt"
)


# 定義完整數字格式
# 可以辨認：
# 4
# 4.462
# -0.0327
# .35
# 1.2e-3
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義已知的類別值
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}

valid_gender_values = {
    "all",
    "male",
    "female"
}

valid_boolean_values = {
    "TRUE",
    "FALSE"
}


# 建立函數：判斷每一個欄位值屬於哪一種型態
def classify_value(value):

    # 去除字串前後空白
    value = value.strip()

    # 空字串
    if value == "":
        return "empty"

    # NA
    if value.upper() == "NA":
        return "NA"

    # 完整純數字
    if full_number_pattern.fullmatch(value):
        return "numeric"

    # 染色體類型
    if value in valid_chr_values:
        return "chromosome"

    # 性別分析群組
    if value in valid_gender_values:
        return "gender"

    # TRUE 或 FALSE
    if value in valid_boolean_values:
        return "boolean"

    # 字串中含有常見解碼異常符號
    if "�" in value or "?" in value:
        return "garbled"

    # 其他文字
    return "text"


# 先開啟檔案取得表頭
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 取得全部欄位名稱
    all_columns = reader.fieldnames


# 排除新增的 left_shift
# 剩下的是原始後段30欄
back_columns = [
    column
    for column in all_columns
    if column != "left_shift"
]


# 每一欄的值統計
column_value_counts = {
    column: Counter()
    for column in back_columns
}

# 每一欄的資料型態統計
column_type_counts = {
    column: Counter()
    for column in back_columns
}

# 每一欄中可轉成數字的值
column_numeric_values = {
    column: []
    for column in back_columns
}


# 用來統計不同的欄位型態模式
pattern_counts = Counter()

# 保存每種模式的前3筆資料列範例
pattern_examples = {}

# left_shift = 2 的總列數
total_shift2 = 0


# 這些欄位先用來建立子模式
# 主要涵蓋 FLD 到 ConversionType
pattern_columns = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO",
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType"
]


# 重新開啟分類後的後段檔案
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只保留 left_shift = 2
        if row["left_shift"] != "2":
            continue

        total_shift2 += 1


        # 逐一檢查後段30欄
        for column in back_columns:

            # 取得此列、此欄的原始值
            value = row[column]

            # 統計該原始值出現次數
            column_value_counts[column][value] += 1

            # 判斷欄位值型態
            value_type = classify_value(value)

            # 統計該型態出現次數
            column_type_counts[column][value_type] += 1

            # 若是純數字，轉成 float 保存
            if value_type == "numeric":
                column_numeric_values[column].append(
                    float(value)
                )


        # 建立這一列的型態模式
        #
        # 例如：
        # garbled、numeric、numeric、numeric...
        pattern_signature = tuple(
            classify_value(row[column])
            for column in pattern_columns
        )

        # 統計此模式出現次數
        pattern_counts[pattern_signature] += 1


        # 保存每種模式前3筆範例
        if pattern_signature not in pattern_examples:
            pattern_examples[pattern_signature] = []

        if len(pattern_examples[pattern_signature]) < 3:
            pattern_examples[pattern_signature].append(
                {
                    "source_row_number": source_row_number,
                    "values": {
                        column: row[column]
                        for column in pattern_columns
                    }
                }
            )


# 建立逐欄摘要報告
with shift2_column_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    # 寫入總列數
    report.write(
        f"left_shift = 2 total rows: "
        f"{total_shift2}\n\n"
    )


    # 逐欄輸出結果
    for column in back_columns:

        report.write("=" * 70 + "\n")
        report.write(f"Column: {column}\n")
        report.write("=" * 70 + "\n")


        # 取得該欄各種型態統計
        type_counts = column_type_counts[column]

        report.write(
            f"numeric: {type_counts['numeric']}\n"
        )

        report.write(
            f"garbled: {type_counts['garbled']}\n"
        )

        report.write(
            f"empty: {type_counts['empty']}\n"
        )

        report.write(
            f"NA: {type_counts['NA']}\n"
        )

        report.write(
            f"chromosome: {type_counts['chromosome']}\n"
        )

        report.write(
            f"gender: {type_counts['gender']}\n"
        )

        report.write(
            f"boolean: {type_counts['boolean']}\n"
        )

        report.write(
            f"other text: {type_counts['text']}\n"
        )


        # 若該欄有數字，輸出數值摘要
        numeric_values = column_numeric_values[column]

        if numeric_values:

            report.write(
                f"numeric minimum: "
                f"{min(numeric_values)}\n"
            )

            report.write(
                f"numeric median: "
                f"{median(numeric_values)}\n"
            )

            report.write(
                f"numeric maximum: "
                f"{max(numeric_values)}\n"
            )


        # 找出前10種最常出現的原始值
        report.write("\nTop 10 original values:\n")

        for value, count in (
            column_value_counts[column].most_common(10)
        ):

            report.write(
                f"{repr(value)}\t{count}\n"
            )

        report.write("\n")


# 建立子模式報告
with shift2_pattern_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        f"left_shift = 2 total rows: "
        f"{total_shift2}\n"
    )

    report.write(
        f"Number of value-type patterns: "
        f"{len(pattern_counts)}\n\n"
    )


    # 依出現次數由多到少顯示前30種模式
    for pattern_rank, (
        pattern_signature,
        pattern_count
    ) in enumerate(
        pattern_counts.most_common(30),
        start=1
    ):

        report.write("=" * 80 + "\n")
        report.write(
            f"Pattern rank: {pattern_rank}\n"
        )

        report.write(
            f"Count: {pattern_count}\n"
        )

        report.write("-" * 80 + "\n")


        # 顯示每一欄對應的值型態
        for column, value_type in zip(
            pattern_columns,
            pattern_signature
        ):

            report.write(
                f"{column}: {value_type}\n"
            )


        # 顯示此模式的前3筆實際範例
        report.write("\nExamples:\n")

        for example in pattern_examples[
            pattern_signature
        ]:

            report.write(
                f"source_row_number: "
                f"{example['source_row_number']}\n"
            )

            for column in pattern_columns:

                value = example["values"][column]

                report.write(
                    f"  {column}: {repr(value)}\n"
                )

            report.write("\n")


# 顯示完成訊息
print("left_shift = 2 逐欄分析完成")
print()
print("left_shift = 2 總列數：", total_shift2)
print("發現的子模式數：", len(pattern_counts))
print()
print("逐欄摘要報告：", shift2_column_report_path)
print("子模式摘要報告：", shift2_pattern_report_path)

left_shift = 2 逐欄分析完成

left_shift = 2 總列數： 3296
發現的子模式數： 2

逐欄摘要報告： /Users/sheena/Desktop/summerintern/split_data/shift2_column_summary.txt
子模式摘要報告： /Users/sheena/Desktop/summerintern/split_data/shift2_subpattern_summary.txt


位移兩個：要分段除理
1.先處理nminor allele, nclus是缺失的 以及位移 
2.後面30欄（NC.meanX 到 Call Modified

In [ ]:
# 載入處理 Tab 分隔檔案的工具
import csv

# 載入正規表示式工具
# 用來從含亂碼的字串尾端擷取數值
import re

# 設定第 1 階段修正後的輸出檔案
# 此檔只清理 FLD 與 HomFLD，不移動其他欄位
shift2_stage1_path = (
    output_folder
    / "genotype_back_shift2_stage1_clean_FLD_HomFLD.txt"
)


# 設定修正前後對照紀錄檔
shift2_stage1_audit_path = (
    output_folder
    / "genotype_back_shift2_stage1_audit.txt"
)


# 定義「完整純數字」格式
# 可辨認：
# 0.548
# -0.0327
# 12
# .35
# 1.2e-3
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義「字串尾端數字」格式
# 例如：
# '?�數???�數??0.548' → 擷取 0.548
# '?�數??-0.0327'     → 擷取 -0.0327
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立計數器
total_shift2 = 0

# FLD 成功擷取的列數
fld_success_count = 0

# FLD 無法擷取的列數
fld_failed_count = 0

# HomFLD 原本就是純數字的列數
homfld_already_numeric_count = 0

# HomFLD 從亂碼尾端成功擷取的列數
homfld_extracted_count = 0

# HomFLD 無法擷取的列數
homfld_failed_count = 0


# 保存少量失敗範例
failed_examples = []


# 開啟：
# 1. 已加入 left_shift 的後段資料
# 2. 第 1 階段修正版
# 3. 修正前後對照紀錄
with classified_back_path.open(
    "r",                         # 讀取模式
    encoding="utf-8",            # 使用 UTF-8
    errors="replace",            # 無法解碼字元以 � 保留
    newline=""
) as input_file, \
shift2_stage1_path.open(
    "w",                         # 建立新的修正版
    encoding="utf-8",
    newline=""
) as output_file, \
shift2_stage1_audit_path.open(
    "w",                         # 建立修正紀錄檔
    encoding="utf-8",
    newline=""
) as audit_file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 建立修正版寫入器
    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 建立對照紀錄寫入器
    audit_writer = csv.writer(
        audit_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )


    # 讀取原始表頭
    header = next(reader)


    # 修正版增加 source_row_number 與 repair_status
    writer.writerow(
        ["source_row_number"]
        + header
        + ["repair_status"]
    )


    # 對照檔只保留這次修復相關的欄位
    audit_writer.writerow(
        [
            "source_row_number",
            "original_FLD",
            "fixed_FLD",
            "original_HomFLD",
            "fixed_HomFLD",
            "repair_status"
        ]
    )


    # 逐列讀取資料
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 第一欄是 left_shift
        # 只處理 left_shift = 2
        if row[0] != "2":
            continue

        total_shift2 += 1


        # 複製整列資料
        # 避免直接修改原始 row
        corrected_row = row.copy()


        # 找出 FLD 與 HomFLD 在表頭中的位置
        fld_index = header.index("FLD")
        homfld_index = header.index("HomFLD")


        # 取得原始值
        original_fld = row[fld_index]
        original_homfld = row[homfld_index]


        # -------------------------
        # 清理 FLD
        # -------------------------

        # 從 FLD 字串尾端擷取數字
        fld_match = ending_number_pattern.search(
            original_fld.strip()
        )

        if fld_match:

            # 取得擷取出的 FLD 數值
            fixed_fld = fld_match.group(1)

            # 放回原本 FLD 欄位
            corrected_row[fld_index] = fixed_fld

            fld_success_count += 1

        else:

            # 無法擷取時保留原始內容
            fixed_fld = original_fld

            fld_failed_count += 1


        # -------------------------
        # 清理 HomFLD
        # -------------------------

        # 如果 HomFLD 本身已是完整純數字
        if full_number_pattern.fullmatch(
            original_homfld.strip()
        ):

            # 保留原始純數字
            fixed_homfld = original_homfld.strip()

            homfld_already_numeric_count += 1

        else:

            # 若 HomFLD 含亂碼，從尾端擷取數字
            homfld_match = ending_number_pattern.search(
                original_homfld.strip()
            )

            if homfld_match:

                # 取得尾端數字
                fixed_homfld = homfld_match.group(1)

                homfld_extracted_count += 1

            else:

                # 無法擷取時保留原始內容
                fixed_homfld = original_homfld

                homfld_failed_count += 1


        # 將清理後的 HomFLD 放回原欄位
        corrected_row[homfld_index] = fixed_homfld


        # 判斷本列是否成功
        if (
            fld_match is not None
            and (
                full_number_pattern.fullmatch(
                    fixed_homfld.strip()
                )
                is not None
            )
        ):
            repair_status = "success"

        else:
            repair_status = "failed"


        # 保存少量失敗範例
        if (
            repair_status == "failed"
            and len(failed_examples) < 20
        ):
            failed_examples.append(
                {
                    "source_row_number": source_row_number,
                    "original_FLD": original_fld,
                    "fixed_FLD": fixed_fld,
                    "original_HomFLD": original_homfld,
                    "fixed_HomFLD": fixed_homfld
                }
            )


        # 寫入第 1 階段修正版
        writer.writerow(
            [source_row_number]
            + corrected_row
            + [repair_status]
        )


        # 寫入修正前後對照
        audit_writer.writerow(
            [
                source_row_number,
                original_fld,
                fixed_fld,
                original_homfld,
                fixed_homfld,
                repair_status
            ]
        )


# 顯示處理結果
print("left_shift = 2，第 1 階段處理完成")
print()

print("修正版檔案：", shift2_stage1_path)
print("修正對照檔：", shift2_stage1_audit_path)
print()

print("left_shift = 2 總列數：", total_shift2)
print("FLD 成功擷取：", fld_success_count)
print("FLD 無法擷取：", fld_failed_count)
print()

print(
    "HomFLD 原本為純數字：",
    homfld_already_numeric_count
)

print(
    "HomFLD 從亂碼成功擷取：",
    homfld_extracted_count
)

print(
    "HomFLD 無法擷取：",
    homfld_failed_count
)

print()
print("失敗範例：")

for example in failed_examples:
    print(example)

left_shift = 2，第 1 階段處理完成

修正版檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_stage1_clean_FLD_HomFLD.txt
修正對照檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_stage1_audit.txt

left_shift = 2 總列數： 3296
FLD 成功擷取： 3296
FLD 無法擷取： 0

HomFLD 原本為純數字： 3058
HomFLD 從亂碼成功擷取： 238
HomFLD 無法擷取： 0

失敗範例：


In [24]:
# 載入 csv 模組，用來讀取 Tab 分隔檔案
import csv

# 載入 re 模組，用來判斷欄位是否為完整數字
import re


# 定義完整數字格式
# 可接受：
# 0.548
# -0.0327
# 12
# .35
# 1.2e-3
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 建立檢查計數器
total_rows = 0                 # 總資料列數
fld_numeric_count = 0          # FLD 為純數字的列數
homfld_numeric_count = 0       # HomFLD 為純數字的列數
both_numeric_count = 0         # FLD 與 HomFLD 都為純數字
repair_success_count = 0       # repair_status 為 success

# 保存前 20 筆仍有問題的資料
problem_examples = []

# 保存前 5 筆清理後範例
clean_examples = []


# 開啟第 1 階段清理後的檔案
with shift2_stage1_path.open(
    "r",                       # 讀取模式
    encoding="utf-8",          # 使用 UTF-8
    errors="replace",          # 無法解碼字元以替代符號保留
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        total_rows += 1

        # 取得清理後的 FLD 與 HomFLD
        fld_value = row["FLD"].strip()
        homfld_value = row["HomFLD"].strip()

        # 判斷 FLD 是否為完整純數字
        fld_is_numeric = bool(
            full_number_pattern.fullmatch(fld_value)
        )

        # 判斷 HomFLD 是否為完整純數字
        homfld_is_numeric = bool(
            full_number_pattern.fullmatch(homfld_value)
        )

        # 分別累計
        if fld_is_numeric:
            fld_numeric_count += 1

        if homfld_is_numeric:
            homfld_numeric_count += 1

        # 兩欄都為純數字
        if fld_is_numeric and homfld_is_numeric:
            both_numeric_count += 1

        # 檢查程式先前標記的修復狀態
        if row["repair_status"] == "success":
            repair_success_count += 1

        # 保存前 5 筆正常範例
        if (
            fld_is_numeric
            and homfld_is_numeric
            and len(clean_examples) < 5
        ):
            clean_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],
                    "FLD": fld_value,
                    "HomFLD": homfld_value,
                    "specialSNP_chr":
                        row["specialSNP_chr"],
                    "gender_metrics":
                        row["gender_metrics"]
                }
            )

        # 保存前 20 筆仍有問題的範例
        if (
            not fld_is_numeric
            or not homfld_is_numeric
        ):
            if len(problem_examples) < 20:
                problem_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],
                        "FLD": fld_value,
                        "HomFLD": homfld_value,
                        "repair_status":
                            row["repair_status"]
                    }
                )


# 顯示整體驗證結果
print("left_shift = 2，第 1 階段驗證結果")
print()

print("總列數：", total_rows)
print("FLD 為純數字：", fld_numeric_count)
print("HomFLD 為純數字：", homfld_numeric_count)
print("FLD 與 HomFLD 都為純數字：", both_numeric_count)
print("repair_status = success：", repair_success_count)

print()
print("前 5 筆清理後範例：")

for example in clean_examples:
    print(example)

print()
print("仍有問題的範例：")

for example in problem_examples:
    print(example)

left_shift = 2，第 1 階段驗證結果

總列數： 3296
FLD 為純數字： 3296
HomFLD 為純數字： 3296
FLD 與 HomFLD 都為純數字： 3296
repair_status = success： 3296

前 5 筆清理後範例：
{'source_row_number': '88', 'FLD': '0.548', 'HomFLD': '3.205', 'specialSNP_chr': 'PolyHighResolution', 'gender_metrics': '1'}
{'source_row_number': '141', 'FLD': '0.43', 'HomFLD': '2.379', 'specialSNP_chr': 'PolyHighResolution', 'gender_metrics': '1'}
{'source_row_number': '459', 'FLD': '0.702', 'HomFLD': '1.583', 'specialSNP_chr': 'NoMinorHom', 'gender_metrics': '1'}
{'source_row_number': '843', 'FLD': '8.928', 'HomFLD': '0.723', 'specialSNP_chr': 'PolyHighResolution', 'gender_metrics': '1'}
{'source_row_number': '932', 'FLD': '0.441', 'HomFLD': '2.365', 'specialSNP_chr': 'NoMinorHom', 'gender_metrics': '1'}

仍有問題的範例：


In [25]:
# 載入 Counter
# 用來統計不同群組、ConversionType 與 genotype 總數
from collections import Counter

# 設定不同 gender_metrics 對應的預期樣本數
expected_total_by_gender = {
    "all": 240,       # 全部240位樣本
    "female": 163,    # 女性163位
    "male": 77        # 男性77位
}

# 合理的染色體分類
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}

# 合理的 gender_metrics
valid_gender_values = {
    "all",
    "female",
    "male"
}

# 合理的 hemizygous 值
valid_hemizygous_values = {
    "0",
    "1"
}


# 建立檢查計數器
total_rows = 0                    # left_shift = 2 總列數
structure_pass_count = 0          # 文字欄位結構合理
genotype_qc_pass_count = 0        # genotype count 合計合理
genotype_qc_fail_count = 0        # genotype count 合計不合理
best_columns_pass_count = 0       # BestProbeset等欄位合理

# 統計不同 genotype count 合計值
batch_total_counts = Counter()

# 統計不同候選資料群組
group_counts = Counter()

# 統計候選 ConversionType
conversion_counts = Counter()

# 保存前20筆未通過範例
failed_examples = []


# 開啟位移2、第1階段清理後的檔案
with shift2_stage1_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼字元以替代符號保留
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        total_rows += 1

        # -------------------------------------------------
        # 假設缺少 nMinorAllele 與 Nclus
        # 因此目前這些欄位中的值應向右移兩格
        # -------------------------------------------------

        # 目前 nMinorAllele 的值，候選對應真正的 n_AA
        candidate_n_AA = row["nMinorAllele"]

        # 目前 Nclus 的值，候選對應真正的 n_AB
        candidate_n_AB = row["Nclus"]

        # 目前 n_AA 的值，候選對應真正的 n_BB
        candidate_n_BB = row["n_AA"]

        # 目前 n_AB 的值，候選對應真正的 n_NC
        candidate_n_NC = row["n_AB"]

        # 目前 n_BB 的值，候選對應真正的 hemizygous
        candidate_hemizygous = row["n_BB"]

        # 目前 n_NC 的值，候選對應真正的 specialSNP_chr
        candidate_chr = row["n_NC"]

        # 目前 hemizygous 的值，候選對應真正的 gender_metrics
        candidate_gender = row["hemizygous"]

        # 目前 specialSNP_chr 的值，候選對應真正的 ConversionType
        candidate_conversion = row["specialSNP_chr"]

        # 目前 gender_metrics 的值，候選對應真正的 BestProbeset
        candidate_best_probeset = row["gender_metrics"]

        # 目前 ConversionType 的值，候選對應真正的 BestandRecommended
        candidate_best_recommended = row["ConversionType"]

        # 目前 BestProbeset 的值，候選對應真正的 HomHet
        candidate_homhet = row["BestProbeset"]


        # -------------------------------------------------
        # 檢查文字欄位位置是否合理
        # -------------------------------------------------

        structure_ok = (
            candidate_hemizygous in valid_hemizygous_values
            and candidate_chr in valid_chr_values
            and candidate_gender in valid_gender_values
            and candidate_conversion
            in {"PolyHighResolution", "NoMinorHom"}
        )

        if structure_ok:
            structure_pass_count += 1


        # -------------------------------------------------
        # 檢查 BestProbeset 等欄位是否合理
        # -------------------------------------------------

        best_columns_ok = (
            candidate_best_probeset in {"0", "1"}
            and candidate_best_recommended in {"0", "1"}
            and candidate_homhet in {"0", "1"}
        )

        if best_columns_ok:
            best_columns_pass_count += 1


        # -------------------------------------------------
        # 統計候選群組
        # -------------------------------------------------

        group_counts[
            (
                candidate_chr,
                candidate_gender,
                candidate_conversion
            )
        ] += 1

        conversion_counts[
            candidate_conversion
        ] += 1


        # -------------------------------------------------
        # 檢查 genotype count
        # -------------------------------------------------

        try:
            # 將四個候選 count 轉成整數
            candidate_counts = [
                int(candidate_n_AA),
                int(candidate_n_AB),
                int(candidate_n_BB),
                int(candidate_n_NC)
            ]

            # 計算四個 genotype count 合計
            batch_total = sum(candidate_counts)

            # 統計實際合計值
            batch_total_counts[batch_total] += 1

            # 根據 candidate_gender 決定合理樣本數
            expected_total = expected_total_by_gender.get(
                candidate_gender
            )

            # 實際總數符合該群組預期樣本數
            count_ok = (
                expected_total is not None
                and batch_total == expected_total
            )

        except (ValueError, TypeError):

            # 無法轉為整數
            batch_total = None
            expected_total = expected_total_by_gender.get(
                candidate_gender
            )
            count_ok = False


        # 通過 genotype count QC
        if count_ok:
            genotype_qc_pass_count += 1

        # 未通過 genotype count QC
        else:
            genotype_qc_fail_count += 1

            # 最多保存前20筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "candidate_n_AA":
                            candidate_n_AA,

                        "candidate_n_AB":
                            candidate_n_AB,

                        "candidate_n_BB":
                            candidate_n_BB,

                        "candidate_n_NC":
                            candidate_n_NC,

                        "batch_total":
                            batch_total,

                        "expected_total":
                            expected_total,

                        "candidate_hemizygous":
                            candidate_hemizygous,

                        "candidate_chr":
                            candidate_chr,

                        "candidate_gender":
                            candidate_gender,

                        "candidate_conversion":
                            candidate_conversion
                    }
                )


# 顯示整體檢查結果
print("left_shift = 2 候選欄位對應驗證")
print()

print("總列數：", total_rows)

print(
    "文字欄位結構合理：",
    structure_pass_count
)

print(
    "BestProbeset相關欄位合理：",
    best_columns_pass_count
)

print(
    "genotype count QC 通過：",
    genotype_qc_pass_count
)

print(
    "genotype count QC 未通過：",
    genotype_qc_fail_count
)


# 顯示 genotype count 合計分布
print()
print("genotype count 合計分布：")

for batch_total, count in batch_total_counts.most_common():
    print(
        "合計 =",
        batch_total,
        "| 列數 =",
        count
    )


# 顯示主要資料群組
print()
print("候選 chromosome／gender／ConversionType 組合：")

for group, count in group_counts.most_common():
    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| ConversionType =",
        group[2],
        "| 列數 =",
        count
    )


# 顯示 ConversionType 分布
print()
print("候選 ConversionType 分布：")

for conversion, count in conversion_counts.most_common():
    print(
        conversion,
        "：",
        count
    )


# 顯示未通過範例
print()
print("前20筆未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2 候選欄位對應驗證

總列數： 3296
文字欄位結構合理： 3296
BestProbeset相關欄位合理： 3296
genotype count QC 通過： 3296
genotype count QC 未通過： 0

genotype count 合計分布：
合計 = 163 | 列數 = 3058
合計 = 240 | 列數 = 238

候選 chromosome／gender／ConversionType 組合：
specialSNP_chr = X | gender_metrics = female | ConversionType = NoMinorHom | 列數 = 1681
specialSNP_chr = X | gender_metrics = female | ConversionType = PolyHighResolution | 列數 = 1377
specialSNP_chr = MT | gender_metrics = all | ConversionType = PolyHighResolution | 列數 = 238

候選 ConversionType 分布：
NoMinorHom ： 1681
PolyHighResolution ： 1615

前20筆未通過範例：


In [ ]:
#n_AA 到 BB.meanY向右修正兩欄
# 設定第 2 階段的輸出檔案
# partial 表示目前只修正到 BB.meanY
shift2_stage2_path = (
    output_folder
    / "genotype_back_shift2_stage2_partial_fixed.txt"
)


# 設定修正前後的對照紀錄檔
shift2_stage2_audit_path = (
    output_folder
    / "genotype_back_shift2_stage2_audit.txt"
)


# 原始欄位中的值，依序應移到右邊兩格的目標欄位
source_fields = [
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY"
]


# 上面 source_fields 中的值，修正後應放入這些欄位
target_fields = [
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY"
]


# 建立處理結果計數器
total_rows = 0
qc_pass_count = 0
qc_fail_count = 0

# 保存前 20 筆未通過檢查的資料
failed_examples = []


# 開啟第 1 階段已清理 FLD、HomFLD 的檔案
with shift2_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift2_stage2_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift2_stage2_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄位名稱讀取資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 保留原始欄位順序
    output_columns = reader.fieldnames

    # 建立修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_columns,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 寫入修正版表頭
    writer.writeheader()


    # 對照檔保存修正前後的關鍵欄位
    audit_columns = (
        ["source_row_number"]
        + ["raw_" + field for field in source_fields]
        + ["fixed_nMinorAllele", "fixed_Nclus"]
        + ["fixed_" + field for field in target_fields]
    )

    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_columns,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 寫入對照檔表頭
    audit_writer.writeheader()


    # 逐列處理 3,296 筆 left_shift = 2
    for row in reader:

        total_rows += 1

        # 先保存所有要搬移的原始值
        # 必須先保存，避免移動過程中覆蓋後面的值
        original_values = {
            field: row[field]
            for field in source_fields
        }


        # nMinorAllele 與 Nclus 沒有可靠的原始值
        # 暫時設為空白，不猜測填值
        row["nMinorAllele"] = ""
        row["Nclus"] = ""


        # 將原始值依照對應關係放入右邊兩格的欄位
        for source_field, target_field in zip(
            source_fields,
            target_fields
        ):
            row[target_field] = original_values[source_field]


        # ----------------------------------------
        # 修正後的合理性檢查
        # ----------------------------------------

        try:
            # 計算修正後四個 genotype count
            batch_total = (
                int(row["n_AA"])
                + int(row["n_AB"])
                + int(row["n_BB"])
                + int(row["n_NC"])
            )

            # 根據 gender_metrics 決定預期樣本數
            expected_total_by_gender = {
                "all": 240,
                "female": 163,
                "male": 77
            }

            expected_total = expected_total_by_gender.get(
                row["gender_metrics"]
            )

            # 同時檢查 count 與文字錨點
            qc_ok = (
                batch_total == expected_total
                and row["hemizygous"] in {"0", "1"}
                and row["specialSNP_chr"]
                in {"autosomal", "X", "Y", "MT"}
                and row["gender_metrics"]
                in {"all", "female", "male"}
                and row["ConversionType"]
                in {"PolyHighResolution", "NoMinorHom"}
                and row["BestProbeset"] in {"0", "1"}
                and row["BestandRecommended"] in {"0", "1"}
                and row["HomHet"] in {"0", "1"}
            )

        except (ValueError, TypeError):
            batch_total = None
            expected_total = None
            qc_ok = False


        # 累計 QC 結果
        if qc_ok:
            qc_pass_count += 1

        else:
            qc_fail_count += 1

            # 保存前 20 筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],
                        "batch_total":
                            batch_total,
                        "expected_total":
                            expected_total,
                        "specialSNP_chr":
                            row["specialSNP_chr"],
                        "gender_metrics":
                            row["gender_metrics"],
                        "ConversionType":
                            row["ConversionType"]
                    }
                )


        # 寫入部分修正後的資料
        writer.writerow(row)


        # 建立修正前後對照紀錄
        audit_row = {
            "source_row_number":
                row["source_row_number"],

            "fixed_nMinorAllele":
                row["nMinorAllele"],

            "fixed_Nclus":
                row["Nclus"]
        }

        # 加入修正前的值
        for field in source_fields:
            audit_row["raw_" + field] = (
                original_values[field]
            )

        # 加入修正後的值
        for field in target_fields:
            audit_row["fixed_" + field] = row[field]

        # 寫入對照紀錄
        audit_writer.writerow(audit_row)


# 顯示結果
print("left_shift = 2，第 2 階段處理完成")
print()

print("輸出檔案：", shift2_stage2_path)
print("對照紀錄：", shift2_stage2_audit_path)
print()

print("總列數：", total_rows)
print("修正後 QC 通過：", qc_pass_count)
print("修正後 QC 未通過：", qc_fail_count)

print()
print("未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2，第 2 階段處理完成

輸出檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_stage2_partial_fixed.txt
對照紀錄： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_stage2_audit.txt

總列數： 3296
修正後 QC 通過： 3296
修正後 QC 未通過： 0

未通過範例：


處理NC.meanX 到 Call Modified

In [27]:
# 載入 csv 模組
# 用來讀取 Tab 分隔的 TXT 檔案
import csv

# 載入正規表示式模組
# 用來判斷某個值是否為純數字
import re

# Counter 用來統計各種欄位排列模式出現的次數
from collections import Counter


# 設定這次要分析的尾端欄位
tail_columns = [
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 設定分析報告的輸出位置
shift2_tail_report_path = (
    output_folder
    / "shift2_tail_pattern_summary.txt"
)


# 定義完整數字格式
# 可辨認：
# 0
# 0.0031
# -0.0327
# 58.27
# 1.32E-05
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 建立函數：判斷欄位值的資料型態
def classify_tail_value(value):

    # 避免值為 None
    if value is None:
        return "missing"

    # 去除字串前後空白
    value = value.strip()

    # 空字串
    if value == "":
        return "empty"

    # NA
    if value.upper() == "NA":
        return "NA"

    # TRUE 或 FALSE
    if value.upper() in {"TRUE", "FALSE"}:
        return "boolean"

    # 完整純數字
    if full_number_pattern.fullmatch(value):
        return "numeric"

    # 出現解碼替代符號或問號
    if "�" in value or "?" in value:
        return "garbled"

    # 其他文字
    return "text"


# 統計尾端欄位型態模式
tail_pattern_counts = Counter()

# 保存每種模式的前 3 筆實際範例
tail_pattern_examples = {}

# 統計模式和染色體、性別、ConversionType、HomRO的關係
group_pattern_counts = Counter()

# 記錄總列數
total_rows = 0


# 開啟位移 2、第 2 階段部分修正後的檔案
with shift2_stage2_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼的字元以 � 顯示
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 確認尾端欄位都存在
    missing_columns = [
        column
        for column in tail_columns
        if column not in reader.fieldnames
    ]

    # 若有欄位不存在，停止執行並顯示錯誤
    if missing_columns:
        raise ValueError(
            f"找不到以下欄位：{missing_columns}"
        )


    # 逐列分析
    for row in reader:

        total_rows += 1


        # 建立此列的尾端欄位型態模式
        #
        # 例如：
        # garbled, numeric, NA, boolean, empty, empty, empty
        pattern_signature = tuple(
            classify_tail_value(row[column])
            for column in tail_columns
        )


        # 統計此模式出現次數
        tail_pattern_counts[pattern_signature] += 1


        # 第一次遇到此模式時建立範例清單
        if pattern_signature not in tail_pattern_examples:
            tail_pattern_examples[pattern_signature] = []


        # 每種模式最多保存前 3 筆實際資料
        if len(tail_pattern_examples[pattern_signature]) < 3:

            tail_pattern_examples[pattern_signature].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "HomRO":
                        row["HomRO"],

                    "specialSNP_chr":
                        row["specialSNP_chr"],

                    "gender_metrics":
                        row["gender_metrics"],

                    "ConversionType":
                        row["ConversionType"],

                    "tail_values": {
                        column: row[column]
                        for column in tail_columns
                    }
                }
            )


        # 同時統計此模式與前面欄位的關係
        group_key = (
            row["specialSNP_chr"],
            row["gender_metrics"],
            row["ConversionType"],
            row["HomRO"],
            pattern_signature
        )

        group_pattern_counts[group_key] += 1


# 將結果寫入報告檔
with shift2_tail_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    # 寫入基本摘要
    report.write(
        f"left_shift = 2 total rows: {total_rows}\n"
    )

    report.write(
        f"Number of tail patterns: "
        f"{len(tail_pattern_counts)}\n\n"
    )


    # 依出現次數由多到少輸出所有模式
    for rank, (
        pattern_signature,
        count
    ) in enumerate(
        tail_pattern_counts.most_common(),
        start=1
    ):

        report.write("=" * 80 + "\n")
        report.write(f"Pattern rank: {rank}\n")
        report.write(f"Count: {count}\n")
        report.write("-" * 80 + "\n")


        # 顯示每個尾端欄位的資料型態
        for column, value_type in zip(
            tail_columns,
            pattern_signature
        ):
            report.write(
                f"{column}: {value_type}\n"
            )


        # 顯示此模式的實際範例
        report.write("\nExamples:\n")

        for example in tail_pattern_examples[
            pattern_signature
        ]:

            report.write(
                f"source_row_number: "
                f"{example['source_row_number']}\n"
            )

            report.write(
                f"HomRO: {repr(example['HomRO'])}\n"
            )

            report.write(
                f"specialSNP_chr: "
                f"{repr(example['specialSNP_chr'])}\n"
            )

            report.write(
                f"gender_metrics: "
                f"{repr(example['gender_metrics'])}\n"
            )

            report.write(
                f"ConversionType: "
                f"{repr(example['ConversionType'])}\n"
            )


            # 顯示尾端 7 欄實際內容
            for column in tail_columns:

                value = example["tail_values"][column]

                report.write(
                    f"  {column}: {repr(value)}\n"
                )

            report.write("\n")


    # 顯示模式與前段特徵的組合
    report.write("\n" + "=" * 80 + "\n")
    report.write("Pattern grouping summary\n")
    report.write("=" * 80 + "\n\n")


    for group_key, count in (
        group_pattern_counts.most_common()
    ):

        chromosome = group_key[0]
        gender = group_key[1]
        conversion = group_key[2]
        homro = group_key[3]
        pattern_signature = group_key[4]

        report.write(
            f"specialSNP_chr={chromosome} | "
            f"gender_metrics={gender} | "
            f"ConversionType={conversion} | "
            f"HomRO={homro} | "
            f"pattern={pattern_signature} | "
            f"count={count}\n"
        )


# 顯示執行結果
print("left_shift = 2 尾端欄位分析完成")
print()

print("總列數：", total_rows)
print("尾端型態模式數：", len(tail_pattern_counts))
print("報告位置：", shift2_tail_report_path)

print()
print("各尾端模式列數：")

for rank, (
    pattern_signature,
    count
) in enumerate(
    tail_pattern_counts.most_common(),
    start=1
):
    print(
        "模式",
        rank,
        "| 列數 =",
        count,
        "|",
        pattern_signature
    )

left_shift = 2 尾端欄位分析完成

總列數： 3296
尾端型態模式數： 2
報告位置： /Users/sheena/Desktop/summerintern/split_data/shift2_tail_pattern_summary.txt

各尾端模式列數：
模式 1 | 列數 = 3190 | ('garbled', 'numeric', 'NA', 'boolean', 'empty', 'empty', 'empty')
模式 2 | 列數 = 106 | ('numeric', 'numeric', 'numeric', 'NA', 'boolean', 'empty', 'empty')


In [28]:
# 指定要查看的尾端欄位
tail_columns = [
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 儲存兩種模式的範例
pattern_examples = {
    1: [],
    2: []
}


# 開啟位移2、第2階段部分修正後的檔案
with shift2_stage2_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼字元以 � 顯示
    newline=""
) as file:

    # 使用欄位名稱讀取資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # 判斷目前屬於哪一種尾端模式
        #
        # 模式1：
        # NC.meanX 含亂碼
        # MMD = NA
        # MinorAlleleFrequency = FALSE
        if (
            ("�" in row["NC.meanX"] or "?" in row["NC.meanX"])
            and row["MMD"] == "NA"
            and row["MinorAlleleFrequency"] in {"TRUE", "FALSE"}
        ):
            pattern_id = 1

        # 模式2：
        # NC.meanX、NC.meanY、MMD 為數字
        # MinorAlleleFrequency = NA
        # H.W.p-Value = FALSE
        elif (
            full_number_pattern.fullmatch(
                row["NC.meanX"].strip()
            )
            and full_number_pattern.fullmatch(
                row["NC.meanY"].strip()
            )
            and full_number_pattern.fullmatch(
                row["MMD"].strip()
            )
            and row["MinorAlleleFrequency"] == "NA"
            and row["H.W.p-Value"] in {"TRUE", "FALSE"}
        ):
            pattern_id = 2

        # 不符合目前兩種模式時先跳過
        else:
            continue


        # 每種模式最多保存5筆
        if len(pattern_examples[pattern_id]) < 5:

            pattern_examples[pattern_id].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "specialSNP_chr":
                        row["specialSNP_chr"],

                    "gender_metrics":
                        row["gender_metrics"],

                    "ConversionType":
                        row["ConversionType"],

                    "values": {
                        column: row[column]
                        for column in tail_columns
                    }
                }
            )


        # 兩種模式都各取得5筆後停止
        if (
            len(pattern_examples[1]) == 5
            and len(pattern_examples[2]) == 5
        ):
            break


# 顯示兩種模式的實際內容
for pattern_id in [1, 2]:

    print("=" * 80)
    print("尾端模式：", pattern_id)
    print("=" * 80)

    for example in pattern_examples[pattern_id]:

        print(
            "source_row_number：",
            example["source_row_number"]
        )

        print(
            "specialSNP_chr：",
            example["specialSNP_chr"]
        )

        print(
            "gender_metrics：",
            example["gender_metrics"]
        )

        print(
            "ConversionType：",
            example["ConversionType"]
        )

        # 顯示尾端欄位內容
        for column in tail_columns:
            print(
                f"{column}: "
                f"{repr(example['values'][column])}"
            )

        print("-" * 80)

尾端模式： 1
source_row_number： 88
specialSNP_chr： X
gender_metrics： female
ConversionType： PolyHighResolution
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
source_row_number： 141
specialSNP_chr： X
gender_metrics： female
ConversionType： PolyHighResolution
BB.meanX: '-2.3795'
BB.meanY: '9.5921'
NC.meanX: '?�數??0.0092'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
source_row_number： 459
specialSNP_chr： X
gender_metrics： female
ConversionType： NoMinorHom
BB.meanX: '-1.5826'
BB.meanY: '10.1036'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
----------

In [29]:
# Counter 用來統計 FALSE 出現在各欄位的次數
from collections import Counter

# 設定要檢查的尾端欄位
tail_columns = [
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]

# 統計 FALSE 所在位置
false_position_counts = Counter()

# 統計 TRUE 所在位置
true_position_counts = Counter()

# 統計每列中布林值出現在哪些欄位
boolean_pattern_counts = Counter()

# 保存每種模式前 3 筆範例
boolean_pattern_examples = {}

# 開啟位移 2、第 2 階段部分修正後的檔案
with shift2_stage2_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # 找出這一列 FALSE 出現在哪些欄位
        false_columns = tuple(
            column
            for column in tail_columns
            if row[column].strip().upper() == "FALSE"
        )

        # 找出這一列 TRUE 出現在哪些欄位
        true_columns = tuple(
            column
            for column in tail_columns
            if row[column].strip().upper() == "TRUE"
        )

        # 分別統計 FALSE 的欄位位置
        for column in false_columns:
            false_position_counts[column] += 1

        # 分別統計 TRUE 的欄位位置
        for column in true_columns:
            true_position_counts[column] += 1

        # 用 FALSE 與 TRUE 的位置組成子模式
        boolean_pattern = (
            false_columns,
            true_columns
        )

        # 統計子模式列數
        boolean_pattern_counts[boolean_pattern] += 1

        # 保存每種子模式前 3 筆範例
        if boolean_pattern not in boolean_pattern_examples:
            boolean_pattern_examples[boolean_pattern] = []

        if len(boolean_pattern_examples[boolean_pattern]) < 3:
            boolean_pattern_examples[boolean_pattern].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "specialSNP_chr":
                        row["specialSNP_chr"],

                    "gender_metrics":
                        row["gender_metrics"],

                    "ConversionType":
                        row["ConversionType"],

                    "values": {
                        column: row[column]
                        for column in tail_columns
                    }
                }
            )


# 顯示 FALSE 各位置的列數
print("FALSE 出現位置統計：")

for column, count in false_position_counts.most_common():
    print(column, "：", count)


# 顯示 TRUE 各位置的列數
print()
print("TRUE 出現位置統計：")

for column, count in true_position_counts.most_common():
    print(column, "：", count)


# 顯示布林值位置子模式
print()
print("布林值位置子模式：")

for rank, (pattern, count) in enumerate(
    boolean_pattern_counts.most_common(),
    start=1
):
    print("=" * 70)
    print("模式：", rank)
    print("列數：", count)
    print("FALSE 欄位：", pattern[0])
    print("TRUE 欄位：", pattern[1])

    # 顯示該模式的實際範例
    for example in boolean_pattern_examples[pattern]:

        print(
            "source_row_number：",
            example["source_row_number"]
        )

        print(
            "specialSNP_chr：",
            example["specialSNP_chr"]
        )

        # 顯示尾端 7 欄原始內容
        for column in tail_columns:
            print(
                f"{column}: "
                f"{repr(example['values'][column])}"
            )

        print("-" * 50)

FALSE 出現位置統計：
MinorAlleleFrequency ： 3190
H.W.p-Value ： 106

TRUE 出現位置統計：

布林值位置子模式：
模式： 1
列數： 3190
FALSE 欄位： ('MinorAlleleFrequency',)
TRUE 欄位： ()
source_row_number： 88
specialSNP_chr： X
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------
source_row_number： 141
specialSNP_chr： X
NC.meanX: '?�數??0.0092'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------
source_row_number： 459
specialSNP_chr： X
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------
模式： 2
列數： 106
FALSE 欄位： ('H.W.p-Value',)
TRUE 欄位： ()
source_row_number： 9558
specialSNP_chr： X
NC.meanX: '41.438'
NC.meanY: '0.0184'
MMD: '0.0457'
MinorAl

In [30]:
# 設定要查看的欄位
tail_check_columns = [
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]

# 分別保存 X 和 MT 的前 3 筆範例
examples = {
    "X": [],
    "MT": []
}

# 必須從 stage1 檔案讀取
# 因為 stage1 只清理 FLD、HomFLD，其他欄位仍保留原始狀態
with shift2_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列讀取
    for row in reader:

        # 在 stage1 尚未修正位移
        # 原始 n_NC 位置放的是實際 specialSNP_chr
        chromosome = row["n_NC"]

        # 只查看 X 和 MT
        if chromosome not in examples:
            continue

        # 每一類最多保存 3 筆
        if len(examples[chromosome]) < 3:
            examples[chromosome].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "values": {
                        column: row[column]
                        for column in tail_check_columns
                    }
                }
            )

        # X 與 MT 都各取得 3 筆後停止
        if (
            len(examples["X"]) == 3
            and len(examples["MT"]) == 3
        ):
            break


# 顯示結果
for chromosome in ["X", "MT"]:

    print("=" * 80)
    print("原始 specialSNP_chr：", chromosome)
    print("=" * 80)

    for example in examples[chromosome]:

        print(
            "source_row_number：",
            example["source_row_number"]
        )

        for column in tail_check_columns:
            print(
                f"{column}: "
                f"{repr(example['values'][column])}"
            )

        print("-" * 80)

原始 specialSNP_chr： X
source_row_number： 88
AB.meanX: 'NA'
AB.meanY: 'NA'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
source_row_number： 141
AB.meanX: '-2.3795'
AB.meanY: '9.5921'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0092'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
source_row_number： 459
AB.meanX: '-1.5826'
AB.meanY: '10.1036'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
原始 specialSNP_chr： MT
source_row_number： 843
A

In [31]:
# 載入處理 Tab 分隔檔案的工具
import csv

# 載入正規表示式工具
# 用來從亂碼字串尾端擷取數字
import re


# 定義完整數字格式
# 可辨認 0、1、0.0123、-0.5、1.2E-5
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義字串尾端數字格式
# 例如亂碼加0.0123會擷取0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立計數器
total_x_rows = 0

# 成功從原始 NC.meanX 擷取候選 MAF
maf_extract_success = 0

# 候選 MAF 落在0到0.5
maf_range_pass = 0

# 候選 H.W.p-Value 落在0到1
hwp_range_pass = 0

# 候選 H.W.chisquared.statistic為NA或非負數
hwchi_pass = 0

# 候選 Call Modified為TRUE或FALSE
call_modified_pass = 0

# NC.meanX與NC.meanY候選值為數字或NA
nc_mean_pass = 0

# 所有條件同時通過
all_conditions_pass = 0

# 保存前20筆未通過範例
failed_examples = []

# 保存前5筆候選對應範例
candidate_examples = []


# 必須讀取stage1檔案
# stage1只清理FLD與HomFLD，其他欄位仍保留原始內容
with shift2_stage1_path.open(
    "r",                    # 以讀取模式開啟
    encoding="utf-8",       # 使用UTF-8
    errors="replace",       # 無法解碼字元以替代符號保留
    newline=""
) as file:

    # 使用欄位名稱讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # stage1尚未移位
        # 原始n_NC欄位中的X才是真正的specialSNP_chr
        if row["n_NC"] != "X":
            continue

        total_x_rows += 1


        # -----------------------------------
        # 候選NC cluster mean
        # -----------------------------------

        # 原始BB.meanX候選對應真正NC.meanX
        candidate_nc_mean_x = row["BB.meanX"].strip()

        # 原始BB.meanY候選對應真正NC.meanY
        candidate_nc_mean_y = row["BB.meanY"].strip()


        # 判斷兩個候選值是否為數字或NA
        nc_x_ok = (
            candidate_nc_mean_x.upper() == "NA"
            or full_number_pattern.fullmatch(
                candidate_nc_mean_x
            )
        )

        nc_y_ok = (
            candidate_nc_mean_y.upper() == "NA"
            or full_number_pattern.fullmatch(
                candidate_nc_mean_y
            )
        )

        nc_mean_ok = bool(nc_x_ok and nc_y_ok)

        if nc_mean_ok:
            nc_mean_pass += 1


        # -----------------------------------
        # 候選MinorAlleleFrequency
        # -----------------------------------

        # 從原始NC.meanX亂碼字串尾端擷取數字
        maf_match = ending_number_pattern.search(
            row["NC.meanX"].strip()
        )

        if maf_match:

            # 取得候選MAF
            candidate_maf = maf_match.group(1)

            maf_extract_success += 1

            # 轉成float檢查值域
            maf_value = float(candidate_maf)

            # Minor allele frequency理論上介於0與0.5
            maf_ok = 0 <= maf_value <= 0.5

        else:

            candidate_maf = ""
            maf_value = None
            maf_ok = False

        if maf_ok:
            maf_range_pass += 1


        # -----------------------------------
        # 候選H.W.p-Value
        # -----------------------------------

        # 原始NC.meanY候選對應真正H.W.p-Value
        candidate_hwp = row["NC.meanY"].strip()

        try:
            hwp_value = float(candidate_hwp)

            # p-value應介於0與1
            hwp_ok = 0 <= hwp_value <= 1

        except ValueError:
            hwp_value = None
            hwp_ok = False

        if hwp_ok:
            hwp_range_pass += 1


        # -----------------------------------
        # 候選H.W.chisquared.statistic
        # -----------------------------------

        # 原始MMD候選對應真正H.W.chisquared.statistic
        candidate_hwchi = row["MMD"].strip()

        # NA視為可接受
        if candidate_hwchi.upper() == "NA":

            hwchi_ok = True

        else:

            try:
                hwchi_value = float(candidate_hwchi)

                # 卡方統計量應為非負值
                hwchi_ok = hwchi_value >= 0

            except ValueError:
                hwchi_ok = False

        if hwchi_ok:
            hwchi_pass += 1


        # -----------------------------------
        # 候選Call Modified
        # -----------------------------------

        # 原始MinorAlleleFrequency候選對應Call Modified
        candidate_call_modified = (
            row["MinorAlleleFrequency"]
            .strip()
            .upper()
        )

        call_modified_ok = (
            candidate_call_modified
            in {"TRUE", "FALSE"}
        )

        if call_modified_ok:
            call_modified_pass += 1


        # -----------------------------------
        # 綜合判斷
        # -----------------------------------

        all_ok = (
            nc_mean_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_modified_ok
        )

        if all_ok:
            all_conditions_pass += 1

        else:

            # 最多保存前20筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "candidate_NC.meanX":
                            candidate_nc_mean_x,

                        "candidate_NC.meanY":
                            candidate_nc_mean_y,

                        "candidate_MAF":
                            candidate_maf,

                        "candidate_H.W.p":
                            candidate_hwp,

                        "candidate_H.W.chi":
                            candidate_hwchi,

                        "candidate_CallModified":
                            candidate_call_modified
                    }
                )


        # 保存前5筆候選修正範例
        if len(candidate_examples) < 5:
            candidate_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "fixed_NC.meanX":
                        candidate_nc_mean_x,

                    "fixed_NC.meanY":
                        candidate_nc_mean_y,

                    "fixed_MMD":
                        "",

                    "fixed_MinorAlleleFrequency":
                        candidate_maf,

                    "fixed_H.W.p-Value":
                        candidate_hwp,

                    "fixed_H.W.chisquared.statistic":
                        candidate_hwchi,

                    "fixed_Call Modified":
                        candidate_call_modified
                }
            )


# 顯示驗證結果
print("left_shift = 2，X chromosome尾端候選對應驗證")
print()

print("X資料總列數：", total_x_rows)
print("候選NC mean合理：", nc_mean_pass)
print("MAF尾端數值擷取成功：", maf_extract_success)
print("候選MAF值域合理：", maf_range_pass)
print("候選H.W.p-Value合理：", hwp_range_pass)
print("候選H.W.chisquared合理：", hwchi_pass)
print("候選Call Modified合理：", call_modified_pass)
print("所有條件同時通過：", all_conditions_pass)

print()
print("前5筆候選修正範例：")

for example in candidate_examples:
    print(example)

print()
print("未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2，X chromosome尾端候選對應驗證

X資料總列數： 3058
候選NC mean合理： 3058
MAF尾端數值擷取成功： 3058
候選MAF值域合理： 2961
候選H.W.p-Value合理： 3058
候選H.W.chisquared合理： 3058
候選Call Modified合理： 2952
所有條件同時通過： 2952

前5筆候選修正範例：
{'source_row_number': '88', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0123', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '141', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0092', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '459', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0123', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '932', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_Mino

In [ ]:
#先修復x染色體
# 載入 Tab 分隔檔案與正規表示式工具
import csv
import re


# 定義完整數字格式
# 可以辨認整數、小數、負數與科學記號
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 建立檢查計數器
total_pattern_b = 0              # X-B模式總列數
nc_mean_pass = 0                 # NC.meanX、NC.meanY合理
mmd_pass = 0                     # 候選MMD合理
maf_pass = 0                     # 候選MAF合理
hwp_pass = 0                     # 候選H.W.p-Value合理
hwchi_pass = 0                   # 候選卡方值合理
call_modified_pass = 0           # 候選Call Modified合理
all_pass = 0                     # 所有條件同時通過

# 保存未通過資料的前20筆
failed_examples = []

# 保存前5筆候選對應
candidate_examples = []


# 必須從stage1讀取
# stage1只清理FLD、HomFLD，其他欄仍是原始位置
with shift2_stage1_path.open(
    "r",                         # 讀取模式
    encoding="utf-8",            # 使用UTF-8
    errors="replace",            # 無法解碼字元以替代符號保留
    newline=""
) as file:

    # 使用欄名讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # stage1中，真正的specialSNP_chr仍位於n_NC
        # 只處理X染色體
        if row["n_NC"] != "X":
            continue

        # X-B模式的特徵：
        # 目前MinorAlleleFrequency為NA
        # 目前H.W.p-Value為TRUE或FALSE
        if not (
            row["MinorAlleleFrequency"].strip().upper() == "NA"
            and row["H.W.p-Value"].strip().upper()
            in {"TRUE", "FALSE"}
        ):
            continue

        total_pattern_b += 1


        # -----------------------------------
        # 候選NC cluster mean
        # -----------------------------------

        # 原始BB.meanX、BB.meanY
        # 候選對應真正的NC.meanX、NC.meanY
        candidate_nc_x = row["BB.meanX"].strip()
        candidate_nc_y = row["BB.meanY"].strip()

        # 數字或NA都視為合理
        nc_x_ok = (
            candidate_nc_x.upper() == "NA"
            or bool(
                full_number_pattern.fullmatch(
                    candidate_nc_x
                )
            )
        )

        nc_y_ok = (
            candidate_nc_y.upper() == "NA"
            or bool(
                full_number_pattern.fullmatch(
                    candidate_nc_y
                )
            )
        )

        nc_ok = nc_x_ok and nc_y_ok

        if nc_ok:
            nc_mean_pass += 1


        # -----------------------------------
        # 候選MMD
        # -----------------------------------

        # 原始NC.meanX候選對應真正MMD
        candidate_mmd = row["NC.meanX"].strip()

        try:
            mmd_value = float(candidate_mmd)

            # 目前先以非負值作為基本合理性檢查
            mmd_ok = mmd_value >= 0

        except ValueError:
            mmd_value = None
            mmd_ok = False

        if mmd_ok:
            mmd_pass += 1


        # -----------------------------------
        # 候選MinorAlleleFrequency
        # -----------------------------------

        # 原始NC.meanY候選對應真正MAF
        candidate_maf = row["NC.meanY"].strip()

        try:
            maf_value = float(candidate_maf)

            # Minor allele frequency應介於0與0.5
            maf_ok = 0 <= maf_value <= 0.5

        except ValueError:
            maf_value = None
            maf_ok = False

        if maf_ok:
            maf_pass += 1


        # -----------------------------------
        # 候選H.W.p-Value
        # -----------------------------------

        # 原始MMD候選對應真正H.W.p-Value
        candidate_hwp = row["MMD"].strip()

        try:
            hwp_value = float(candidate_hwp)

            # p-value應介於0與1
            hwp_ok = 0 <= hwp_value <= 1

        except ValueError:
            hwp_value = None
            hwp_ok = False

        if hwp_ok:
            hwp_pass += 1


        # -----------------------------------
        # 候選H.W.chisquared.statistic
        # -----------------------------------

        # 原始MinorAlleleFrequency候選對應卡方值
        candidate_hwchi = (
            row["MinorAlleleFrequency"].strip()
        )

        # NA可以接受
        if candidate_hwchi.upper() == "NA":
            hwchi_ok = True

        else:
            try:
                hwchi_value = float(candidate_hwchi)

                # 卡方統計量應為非負數
                hwchi_ok = hwchi_value >= 0

            except ValueError:
                hwchi_ok = False

        if hwchi_ok:
            hwchi_pass += 1


        # -----------------------------------
        # 候選Call Modified
        # -----------------------------------

        # 原始H.W.p-Value候選對應Call Modified
        candidate_call_modified = (
            row["H.W.p-Value"].strip().upper()
        )

        call_ok = (
            candidate_call_modified
            in {"TRUE", "FALSE"}
        )

        if call_ok:
            call_modified_pass += 1


        # -----------------------------------
        # 綜合檢查
        # -----------------------------------

        row_all_ok = (
            nc_ok
            and mmd_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_ok
        )

        if row_all_ok:
            all_pass += 1

        else:
            # 最多保存20筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "candidate_NC.meanX":
                            candidate_nc_x,

                        "candidate_NC.meanY":
                            candidate_nc_y,

                        "candidate_MMD":
                            candidate_mmd,

                        "candidate_MAF":
                            candidate_maf,

                        "candidate_H.W.p":
                            candidate_hwp,

                        "candidate_H.W.chi":
                            candidate_hwchi,

                        "candidate_CallModified":
                            candidate_call_modified
                    }
                )


        # 保存前5筆候選對應
        if len(candidate_examples) < 5:
            candidate_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "fixed_NC.meanX":
                        candidate_nc_x,

                    "fixed_NC.meanY":
                        candidate_nc_y,

                    "fixed_MMD":
                        candidate_mmd,

                    "fixed_MinorAlleleFrequency":
                        candidate_maf,

                    "fixed_H.W.p-Value":
                        candidate_hwp,

                    "fixed_H.W.chisquared.statistic":
                        candidate_hwchi,

                    "fixed_Call Modified":
                        candidate_call_modified
                }
            )


# 顯示驗證結果
print("left_shift = 2，X chromosome尾端模式B驗證")
print()

print("模式B總列數：", total_pattern_b)
print("候選NC mean合理：", nc_mean_pass)
print("候選MMD合理：", mmd_pass)
print("候選MAF合理：", maf_pass)
print("候選H.W.p-Value合理：", hwp_pass)
print("候選H.W.chisquared合理：", hwchi_pass)
print("候選Call Modified合理：", call_modified_pass)
print("所有條件同時通過：", all_pass)

print()
print("前5筆候選修正範例：")

for example in candidate_examples:
    print(example)

print()
print("未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2，X chromosome尾端模式B驗證

模式B總列數： 106
候選NC mean合理： 106
候選MMD合理： 106
候選MAF合理： 106
候選H.W.p-Value合理： 106
候選H.W.chisquared合理： 106
候選Call Modified合理： 106
所有條件同時通過： 106

前5筆候選修正範例：
{'source_row_number': '9558', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '41.438', 'fixed_MinorAlleleFrequency': '0.0184', 'fixed_H.W.p-Value': '0.0457', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '11324', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '62.254', 'fixed_MinorAlleleFrequency': '0.0215', 'fixed_H.W.p-Value': '0.0636', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '20011', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '61.486', 'fixed_MinorAlleleFrequency': '0.0215', 'fixed_H.W.p-Value': '0.0636', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '20869', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'f

In [ ]:
# 載入處理 Tab 分隔檔案的工具
import csv

# 載入正規表示式工具
# 用來從 X-A 模式的亂碼字串尾端擷取 MAF
import re


# 設定修復後的 X chromosome 輸出檔案
shift2_x_fixed_path = (
    output_folder
    / "genotype_back_shift2_X_fixed.txt"
)


# 設定修復前後對照紀錄檔
shift2_x_audit_path = (
    output_folder
    / "genotype_back_shift2_X_fixed_audit.txt"
)


# 定義字串尾端數字格式
# 例如：
# '?�數??0.0123' → 0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立處理結果計數器
total_x_rows = 0              # X chromosome 總列數
pattern_xa_count = 0          # X-A 模式列數
pattern_xb_count = 0          # X-B 模式列數
unrecognized_count = 0        # 無法分類的列數
qc_pass_count = 0             # 修復後通過 QC
qc_fail_count = 0             # 修復後未通過 QC


# 保存少量無法辨識或 QC 失敗的範例
problem_examples = []


# 中段欄位對應關係
# key 是修正後欄位
# value 是原始 stage1 中提供資料的欄位
middle_mapping = {
    "n_AA": "nMinorAllele",
    "n_AB": "Nclus",
    "n_BB": "n_AA",
    "n_NC": "n_AB",
    "hemizygous": "n_BB",
    "specialSNP_chr": "n_NC",
    "gender_metrics": "hemizygous",
    "ConversionType": "specialSNP_chr",
    "BestProbeset": "gender_metrics",
    "BestandRecommended": "ConversionType",
    "HomHet": "BestProbeset",
    "AA.meanX": "BestandRecommended",
    "AA.meanY": "HomHet",
    "AB.meanX": "AA.meanX",
    "AB.meanY": "AA.meanY",
    "BB.meanX": "AB.meanX",
    "BB.meanY": "AB.meanY"
}


# 開啟 stage1 檔案
# stage1 只清理 FLD 與 HomFLD，尚未移動其他欄位
with shift2_stage1_path.open(
    "r",                         # 讀取模式
    encoding="utf-8",            # 使用 UTF-8
    errors="replace",            # 無法解碼字元以替代符號保留
    newline=""
) as input_file, \
shift2_x_fixed_path.open(
    "w",                         # 建立 X chromosome 修正版
    encoding="utf-8",
    newline=""
) as output_file, \
shift2_x_audit_path.open(
    "w",                         # 建立修復紀錄檔
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄位名稱讀取資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 保留原始欄位，再新增兩個修復紀錄欄位
    output_fields = (
        reader.fieldnames
        + [
            "shift2_subpattern",
            "final_qc_status"
        ]
    )


    # 建立修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    # 寫入修正版表頭
    writer.writeheader()


    # 對照紀錄檔的欄位
    audit_fields = [
        "source_row_number",
        "shift2_subpattern",
        "raw_NC.meanX",
        "raw_NC.meanY",
        "raw_MMD",
        "raw_MinorAlleleFrequency",
        "raw_H.W.p-Value",
        "fixed_NC.meanX",
        "fixed_NC.meanY",
        "fixed_MMD",
        "fixed_MinorAlleleFrequency",
        "fixed_H.W.p-Value",
        "fixed_H.W.chisquared.statistic",
        "fixed_Call Modified",
        "final_qc_status"
    ]


    # 建立對照檔寫入器
    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 寫入對照檔表頭
    audit_writer.writeheader()


    # 逐列處理 stage1 資料
    for row in reader:

        # stage1 尚未修正位移
        # 真正的 specialSNP_chr 仍放在原始 n_NC
        # 只處理 X chromosome
        if row["n_NC"] != "X":
            continue

        total_x_rows += 1


        # 完整保存修復前的資料
        # 後面所有欄位都從 raw 取值，避免互相覆蓋
        raw = row.copy()


        # 建立修正後資料
        fixed = row.copy()


        # -----------------------------------------
        # 第一部分：nMinorAllele 到 BB.meanY
        # -----------------------------------------

        # nMinorAllele 與 Nclus 沒有可靠輸出
        # 暫時設為空白
        fixed["nMinorAllele"] = ""
        fixed["Nclus"] = ""


        # 依照已驗證的對應關係移動中段欄位
        for target_field, source_field in middle_mapping.items():
            fixed[target_field] = raw[source_field]


        # -----------------------------------------
        # 第二部分：分類 X-A 或 X-B 尾端模式
        # -----------------------------------------

        raw_maf_position = (
            raw["MinorAlleleFrequency"]
            .strip()
            .upper()
        )

        raw_hwp_position = (
            raw["H.W.p-Value"]
            .strip()
            .upper()
        )


        # X-A 模式：
        # 原始 MinorAlleleFrequency 位置為 TRUE/FALSE
        if raw_maf_position in {"TRUE", "FALSE"}:

            shift2_subpattern = "X-A"

            pattern_xa_count += 1


            # 原始 BB.meanX、BB.meanY
            # 修正後對應 NC.meanX、NC.meanY
            fixed["NC.meanX"] = raw["BB.meanX"]
            fixed["NC.meanY"] = raw["BB.meanY"]


            # 此模式沒有可靠 MMD
            fixed["MMD"] = ""


            # 從原始 NC.meanX 亂碼尾端擷取 MAF
            maf_match = ending_number_pattern.search(
                raw["NC.meanX"].strip()
            )

            if maf_match:
                fixed["MinorAlleleFrequency"] = (
                    maf_match.group(1)
                )
            else:
                fixed["MinorAlleleFrequency"] = ""


            # 原始 NC.meanY 對應 H.W.p-Value
            fixed["H.W.p-Value"] = raw["NC.meanY"]


            # 原始 MMD 對應卡方統計量
            fixed["H.W.chisquared.statistic"] = (
                raw["MMD"]
            )


            # 原始 MinorAlleleFrequency 位置的布林值
            # 對應 Call Modified
            fixed["Call Modified"] = raw_maf_position


        # X-B 模式：
        # 原始 H.W.p-Value 位置為 TRUE/FALSE
        elif (
            raw["MinorAlleleFrequency"]
            .strip()
            .upper() == "NA"
            and raw_hwp_position in {"TRUE", "FALSE"}
        ):

            shift2_subpattern = "X-B"

            pattern_xb_count += 1


            # 原始 BB.meanX、BB.meanY
            # 修正後對應 NC.meanX、NC.meanY
            fixed["NC.meanX"] = raw["BB.meanX"]
            fixed["NC.meanY"] = raw["BB.meanY"]


            # 原始 NC.meanX 對應 MMD
            fixed["MMD"] = raw["NC.meanX"]


            # 原始 NC.meanY 對應 MAF
            fixed["MinorAlleleFrequency"] = (
                raw["NC.meanY"]
            )


            # 原始 MMD 對應 H.W.p-Value
            fixed["H.W.p-Value"] = raw["MMD"]


            # 原始 MinorAlleleFrequency 對應卡方值
            fixed["H.W.chisquared.statistic"] = (
                raw["MinorAlleleFrequency"]
            )


            # 原始 H.W.p-Value 的布林值
            # 對應 Call Modified
            fixed["Call Modified"] = raw_hwp_position


        # 不符合 X-A 或 X-B
        else:

            shift2_subpattern = "unrecognized"

            unrecognized_count += 1


        # -----------------------------------------
        # 第三部分：修復後 QC
        # -----------------------------------------

        try:
            # X/female 的 genotype count 應合計為 163
            batch_total = (
                int(fixed["n_AA"])
                + int(fixed["n_AB"])
                + int(fixed["n_BB"])
                + int(fixed["n_NC"])
            )


            # MAF 應介於 0 到 0.5
            maf_value = float(
                fixed["MinorAlleleFrequency"]
            )

            maf_ok = 0 <= maf_value <= 0.5


            # H.W.p-Value 應介於 0 到 1
            hwp_value = float(
                fixed["H.W.p-Value"]
            )

            hwp_ok = 0 <= hwp_value <= 1


            # Call Modified 應為 TRUE 或 FALSE
            call_modified_ok = (
                fixed["Call Modified"]
                .strip()
                .upper()
                in {"TRUE", "FALSE"}
            )


            # 文字欄位與 genotype count 同時驗證
            qc_ok = (
                shift2_subpattern in {"X-A", "X-B"}
                and batch_total == 163
                and fixed["hemizygous"] in {"0", "1"}
                and fixed["specialSNP_chr"] == "X"
                and fixed["gender_metrics"] == "female"
                and fixed["ConversionType"]
                in {
                    "PolyHighResolution",
                    "NoMinorHom"
                }
                and maf_ok
                and hwp_ok
                and call_modified_ok
            )

        except (ValueError, TypeError):

            batch_total = None
            qc_ok = False


        # 設定 QC 結果
        if qc_ok:

            final_qc_status = "pass"

            qc_pass_count += 1

        else:

            final_qc_status = "fail"

            qc_fail_count += 1

            # 最多保存前20筆問題範例
            if len(problem_examples) < 20:
                problem_examples.append(
                    {
                        "source_row_number":
                            raw["source_row_number"],

                        "shift2_subpattern":
                            shift2_subpattern,

                        "batch_total":
                            batch_total,

                        "MAF":
                            fixed.get(
                                "MinorAlleleFrequency",
                                ""
                            ),

                        "H.W.p-Value":
                            fixed.get(
                                "H.W.p-Value",
                                ""
                            ),

                        "Call Modified":
                            fixed.get(
                                "Call Modified",
                                ""
                            )
                    }
                )


        # 將模式與 QC 狀態放入修正版
        fixed["shift2_subpattern"] = (
            shift2_subpattern
        )

        fixed["final_qc_status"] = (
            final_qc_status
        )


        # 寫入修正後資料
        writer.writerow(fixed)


        # 寫入修復前後對照
        audit_writer.writerow(
            {
                "source_row_number":
                    raw["source_row_number"],

                "shift2_subpattern":
                    shift2_subpattern,

                "raw_NC.meanX":
                    raw["NC.meanX"],

                "raw_NC.meanY":
                    raw["NC.meanY"],

                "raw_MMD":
                    raw["MMD"],

                "raw_MinorAlleleFrequency":
                    raw["MinorAlleleFrequency"],

                "raw_H.W.p-Value":
                    raw["H.W.p-Value"],

                "fixed_NC.meanX":
                    fixed.get("NC.meanX", ""),

                "fixed_NC.meanY":
                    fixed.get("NC.meanY", ""),

                "fixed_MMD":
                    fixed.get("MMD", ""),

                "fixed_MinorAlleleFrequency":
                    fixed.get(
                        "MinorAlleleFrequency",
                        ""
                    ),

                "fixed_H.W.p-Value":
                    fixed.get(
                        "H.W.p-Value",
                        ""
                    ),

                "fixed_H.W.chisquared.statistic":
                    fixed.get(
                        "H.W.chisquared.statistic",
                        ""
                    ),

                "fixed_Call Modified":
                    fixed.get(
                        "Call Modified",
                        ""
                    ),

                "final_qc_status":
                    final_qc_status
            }
        )


# 顯示處理結果
print("left_shift = 2，X chromosome正式修復完成")
print()

print("修正版檔案：", shift2_x_fixed_path)
print("修復對照檔：", shift2_x_audit_path)
print()

print("X chromosome總列數：", total_x_rows)
print("X-A模式：", pattern_xa_count)
print("X-B模式：", pattern_xb_count)
print("無法辨識模式：", unrecognized_count)
print("最終QC通過：", qc_pass_count)
print("最終QC未通過：", qc_fail_count)

print()
print("問題範例：")

for example in problem_examples:
    print(example)